Plot the exported highres pickle files of NenuFAR

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import astropy.units as u
from astropy.time import Time
from datetime import datetime, timedelta
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from matplotlib.ticker import AutoMinorLocator
import matplotlib as mpl
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00' # use precise epoch
try: mdates.set_epoch('1970-01-01T00:00:00')
except: pass


data_dir    = '/home/mnedal/data/ORFEES'
folder_path = '/home/mnedal/data/pkl_files'
outputs     = '/home/mnedal/data/png'

In [ ]:
mydate = '20250326'

nenufar_files = sorted(glob.glob(f'{folder_path}/nenufar/*'))
for f in enumerate(nenufar_files):
    print(f)

In [ ]:
SRB_groupname = 'typeII' # typeIII_G1, typeIII_G2, typeIII_G3, typeIII_G4, typeII

g_files = [f for f in nenufar_files if mydate in f and f.endswith(f'{SRB_groupname}.pkl')] # find the filenames that end with G1
print(*g_files, sep='\n')

stokesI_filename  = [f for f in g_files if 'stokesI' in f][0]
stokesVI_filename = [f for f in g_files if 'stokesV_over_I' in f][0]

df_int = pd.read_pickle(stokesI_filename)
df_pol = pd.read_pickle(stokesVI_filename)

# Convert arb. unit to dB for Stokes I
df_int = 10 * np.log10(df_int) # Convert the amplitude to decibels

<b>Original cadence:</b>
* Frequency resolution: 6.10 kHz
* Time resolution: 20.97 ms

In [ ]:
dyspec_subtracted = df_int - np.tile(np.nanmedian(df_int,0), (df_int.shape[0],1))

print('Stokes I:')
print(f'Frequency resolution: {np.nanmedian(np.diff(df_int.columns)*1e3):.2f} kHz')
print(f'Time resolution: {np.nanmedian(np.diff(df_int.index)/np.timedelta64(1,"ms")):.2f} ms')
print(f'Data shape: {df_int.shape}\n')

print('Stokes V/I:')
print(f'Frequency resolution: {np.nanmedian(np.diff(df_pol.columns)*1e3):.2f} kHz')
print(f'Time resolution: {np.nanmedian(np.diff(df_pol.index)/np.timedelta64(1,"ms")):.2f} ms')
print(f'Data shape: {df_pol.shape}')

## Plot the highres dynamic spectra

In [ ]:
fig = plt.figure(figsize=[12,5])

ax = fig.add_subplot(121)
pc = ax.pcolormesh(dyspec_subtracted.index, dyspec_subtracted.columns, dyspec_subtracted.T,
                   vmin=0, vmax=np.percentile(dyspec_subtracted, 99),
                   cmap='Spectral_r')
fig.colorbar(pc, ax=ax, pad=0.02, label='dB (background subtracted)')
ax.set_xlabel(f'Time (UT) on {df_int.index[0].date()}')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlim(left=pd.Timestamp(f'{df_int.index[0].date()} 09:36:00'),
#             right=pd.Timestamp(f'{df_int.index[0].date()} 09:41:00'))

ax = fig.add_subplot(122)
pc = ax.pcolormesh(df_pol.index, df_pol.columns, df_pol.T,
                   vmin=-1, vmax=1,
                   cmap='seismic')
fig.colorbar(pc, ax=ax, pad=0.02, label='Polarization fraction')
ax.set_xlabel(f'Time (UT) on {df_pol.index[0].date()}')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlim(left=pd.Timestamp(f'{df_pol.index[0].date()} 09:36:00'),
#             right=pd.Timestamp(f'{df_pol.index[0].date()} 09:41:00'))
fig.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=[15,5])
ax = fig.add_subplot()
ax.pcolormesh(dyspec_subtracted.index, dyspec_subtracted.columns, dyspec_subtracted.T,
              vmin=0, vmax=np.nanpercentile(dyspec_subtracted, 98), cmap='Spectral_r')
ax.set_xlabel(f'Time (UT) on {df_int.index[0].date()}')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlim(left=pd.Timestamp(f'{df_int.index[0].date()} 09:36:00'),
#             right=pd.Timestamp(f'{df_int.index[0].date()} 09:41:00'))
fig.tight_layout()
plt.show()

In [ ]:
print(np.diff(dyspec_subtracted.index[:4]))
print([n*1e-9 for n in np.diff(dyspec_subtracted.index[:4])])

In [ ]:
print(round(20972000*1e-9, 2))
(round(20972000*1e-9, 2) * 1e3)

---

# Full shock analysis of the type II radio burst &mdash; 26 March 2025 (NenuFAR)

Everything above loads and displays the NenuFAR high-resolution Stokes&nbsp;I and Stokes&nbsp;V/I
dynamic spectra. Everything below turns the type II burst in those spectra into a set of shock
characteristics.

**The burst.** One type II seen in both plasma-emission harmonics. The
**fundamental** ($s=1$) enters the NenuFAR window at ~09:20&nbsp;UT near 73&nbsp;MHz and drifts
down until it leaves the bottom of the band around 09:33. The **harmonic** ($s=2$) sits at twice
the fundamental frequency, so it only enters the top of the band at ~09:26, once
$2f_F\lesssim85$&nbsp;MHz, and then follows the same shock out to ~09:50 at ~33&nbsp;MHz. The two
bands therefore appear offset along the time axis even though they are simultaneous &mdash; each is
visible only over the interval where its own frequency falls inside the observing window, and they
overlap around 09:26&ndash;09:33 where $f_H/f_F=2$ can be checked directly. Each band shows its own
band splitting and fine structure.

**Tracing.** There is **one interactive plot of the whole dynamic spectrum** and nothing is split
or windowed. You click along a lane, press **End trace**, and it is recorded under the current
label &mdash; `F lane 1`, `F lane 2`, `H lane 1`, and so on. Trace as many lanes per band as the
burst shows; the analysis works out which is which afterwards by sorting them in frequency, so the
lowest-frequency lane of a band is taken as the upstream branch of its split and the next as the
downstream branch. Nothing about the split has to be declared in advance.

**What the chain does**

1. Takes `dyspec_subtracted` as the tracing layer, restricted to the type II window and decimated
   in time so one interactive figure stays responsive.
2. You trace the lanes on that single figure; each lane can be traced more than once, and the
   scatter between repeats becomes an error bar.
3. Fits each lane $f(t)$ with a low-order polynomial in $\log_{10}f$, keeping the coefficient
   covariance, and Monte-Carlo samples it so the **fit error and the repeat error propagate
   together**.
4. Per band the split gives the density jump and the perpendicular-shock Alfv&eacute;n Mach number
$$X=\left(\frac{f_U}{f_L}\right)^2,\qquad
  M_A=\sqrt{\frac{X\,(X+5)}{2\,(4-X)}},$$
   both of which are **independent of the coronal density model** and are measured twice over, once
   from each band.
5. Each lane frequency is converted to a source height with its own harmonic number through five
   electron-density models at folds 1&ndash;4. Because $f_H=2f_F$ maps to the same local density,
   the two bands give **one continuous height-time track** covering the whole burst &mdash; longer
   than either band alone spans. Differentiating $r(t)$ gives $v_{\rm sh}$ and $a$, and
$$v_A=\frac{v_{\rm sh}}{M_A},\qquad B=v_A\sqrt{\mu_0\rho},\qquad \rho=\mu\, m_p\, n_e .$$
6. &sect;A.5 checks the fundamental/harmonic identification against the data: the frequency ratio
   over the overlap, the relative drift rates, the agreement of the two height tracks, and the
   contrast in circular polarisation between the bands from the **Stokes V/I** frame.
7. The joint height-time track is refit three ways &mdash; polynomial, Gallagher et&nbsp;al. (2003)
   and Byrne et&nbsp;al. (2013) &mdash; with bootstrap error bands, to show the kinematics are not
   an artefact of one fitting choice.
8. Everything is exported as CSV + LaTeX tables plus a drafted results paragraph.

## A.0 &middot; Imports, constants and configuration

In [ ]:
# only what the shock analysis adds; everything else comes from the import cell at the top
import pickle
import textwrap
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import CubicSpline
from matplotlib.colors import Normalize, PowerNorm
from astropy.constants import R_sun, m_e, m_p, e, eps0, mu0, c

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

# physical constants taken from astropy rather than hardcoded
R_SUN_M    = R_sun.to('m').value
C_MS       = c.to('m/s').value
E_CHARGE_J = e.si.value
M_E        = m_e.value
M_P        = m_p.value
EPS0       = eps0.value
MU0        = mu0.value

# electron plasma frequency: f_p [Hz] = PLASMA_CONST * sqrt(n_e [cm^-3])
PLASMA_CONST = (1 / (2 * np.pi)) * np.sqrt(1e6 * E_CHARGE_J**2 / (EPS0 * M_E))
print(f'PLASMA_CONST = {PLASMA_CONST:.4g}   (f_pe [Hz] = PLASMA_CONST * sqrt(n_e [cm^-3]))')

In [ ]:
# ============================== CONFIGURATION ==============================
EVENT_DATE = str(df_int.index[0].date())
OUTDIR     = os.path.join(outputs, 'type2_nenufar')
os.makedirs(OUTDIR, exist_ok=True)
# a marker file stamped now: A.10 compares file times against it with the same clock and the
# same call, so the stale-output check cannot be thrown off by timezone or epoch conventions
_marker = os.path.join(OUTDIR, '.run_marker')
open(_marker, 'w').close()
RUN_START = os.path.getmtime(_marker)

# --- emission physics ---------------------------------------------------------
HARM = {'F': 1, 'H': 2}          # harmonic number s per band
BAND_NAME = {'F': 'Fundamental', 'H': 'Harmonic'}
BANDS = ['F', 'H']               # bands offered by the tracer, in this order
MU   = 1.27                      # mean molecular weight per electron (10% He corona)

# --- tracing ------------------------------------------------------------------
# Lanes are not declared in advance: you trace them on one plot and each finished trace is
# recorded as 'F lane 1', 'F lane 2', 'H lane 1', ... Within a band the lanes are sorted by
# frequency afterwards, so the lowest is taken as the upstream branch of the split and the next
# as the downstream branch. Trace each lane N_REPS times to get a reproducibility error bar.
N_REPS = 3

TRACE_METHOD = 'click'           # which method the tracer opens with; switch it live in the
                                 # widget at any time. 'click'  -> click points along the lane
                                 #                     'bezier' -> place a Bezier curve on it

# --- Bezier tracing -----------------------------------------------------------
BEZIER_ANCHORS    = 2            # 1 -> quadratic (1 anchor), 2 -> cubic (2 anchors). Cubic by
                                 # default: these lanes span close to an octave and a quadratic
                                 # cannot follow them closely enough to recover the band split
BEZIER_NUM_POINTS = 80           # samples taken along the curve when it is recorded
BEZIER_JITTER     = 1.5          # sigma of the control-point jitter used for the auto-repeats,
                                 # in spectrogram pixels; repeat 0 is always the curve you placed
BEZIER_SEED       = 0

TYPEII_WINDOW = (pd.Timestamp(f'{EVENT_DATE} 09:18:00'), pd.Timestamp(f'{EVENT_DATE} 09:52:00'))
TYPEII_FLIM   = [25, 85]         # MHz, the part of the NenuFAR band the burst occupies

# --- decimation of the tracing layer -----------------------------------------
# the exported grid (~123k x 640 for this event) is far finer in time than the tracing needs and
# too large to percentile-scale and pcolormesh interactively, so the layer is block-averaged down
# to at most this size. Leave LAYER_MAX_F generous: the band split and the fine structure live
# in frequency, so nothing should be averaged there
LAYER_MAX_T = 4000               # time samples kept in the tracing layer
LAYER_MAX_F = 1200               # frequency channels kept

# --- display stretch of the tracing layer -------------------------------------
# Keep LAYER_PLO low: clipping the bottom of the distribution is what erases the faint
# background and the fine structure. LAYER_GAMMA < 1 applies a power-law stretch that brightens
# the faint end without throwing it away, which is the gentler way to bring detail up.
LAYER_MODE   = 'db_sub'          # 'db_sub' -> dyspec_subtracted, i.e. dB above the background
                                 # 'ratio'  -> linear ratio to the background, log colour scale
LAYER_CMAP   = 'Spectral_r'
LAYER_PLO    = 2                 # lower percentile of the colour scale
LAYER_PHI    = 98                # upper percentile of the colour scale
LAYER_GAMMA  = 0.6               # <1 brightens faint features, 1 = plain linear stretch
INVERT_FREQ  = True              # low frequency at the top, as in the figures above
PREVIEW_MAX_TCOLS = 1600         # further time down-sampling for the previews only

# --- density-model grid -------------------------------------------------------
FOLDS = [1, 2, 3, 4]
REF_MODEL_NAME = 'Newkirk x2'    # model x fold used for the reference figures and tables
R_BOUNDS = (1.0, 5.0)            # heliocentric range over which n_e(r) is inverted

# --- what is analysed ----------------------------------------------------------
# Every traced lane is analysed on its own. MAKE_JOINT additionally builds an 'F+H joint' track
# that averages the two bands' UPSTREAM lanes where they overlap and takes whichever exists
# elsewhere. That buys time coverage (a second derivative needs a long baseline) but it mixes two
# independent measurements into one curve, so it is off by default.
MAKE_JOINT = False
REF_LANE   = None                # lane the height-time fits and the model sweep use.
                                 # None -> the upstream lane of the first band traced

# --- Monte-Carlo and height-time fitting ---------------------------------------
N_MC      = 100                  # MC draws per pass for the fit error
POLY_DEG  = 2                    # polynomial degree for f(t) and for h(t)
KIN_MIN_PTS = 6                  # minimum finite height points before v and a are computed
KIN_MIN_BASELINE_S = 600         # a QUADRATIC r(t) needs a long enough baseline for its curvature
                                 # to mean anything. Below this a lane is fitted with a straight
                                 # line instead: you can measure a speed over 4 minutes, you
                                 # cannot measure an acceleration, and a degenerate quadratic
                                 # returns whatever curvature the noise asks for - which is how
                                 # two branches of one split ended up with opposite accelerations
KIN_MIN_FRAC = 0.5               # of a lane's OWN traced span the density model must resolve.
                                 # Judged against the lane's span, not the shared grid: a short
                                 # lane is a legitimate measurement, it just covers less time

# --- how the frequency uncertainty of a traced lane is set -----------------------
# A Bezier is sampled at BEZIER_NUM_POINTS along a curve with only ~8 free parameters, so those
# samples are nowhere near independent and np.polyfit's covariance comes out far too small.
# Thinning to one point per FIT_MIN_DT_S seconds removes the fake sample count, and giving each
# point an explicit frequency error (from the QC offset from the ridge) makes the covariance
# mean something rather than just tracking how smooth the curve happens to be.
FIT_MIN_DT_S    = 10             # s, minimum spacing of the points a lane fit actually uses
LANE_SIGMA_MHZ  = 'auto'         # 'auto' -> per-lane rms offset from the ridge (A.3 QC),
                                 # a number -> use that in MHz, None -> old behaviour
LANE_SIGMA_FLOOR = 0.2           # MHz, floor on that uncertainty

# --- sanity thresholds used by the consistency audit in A.5 ----------------------
QC_MIN_SNR_DB   = 3.0            # a trace sitting on less than this above background is not on
                                 # the burst. The offset-from-ridge check alone cannot catch that:
                                 # over empty sky the brightest pixel within the search window is
                                 # noise, which lands near zero offset by chance
CONSIST_SIGMA   = 3.0            # flag a disagreement beyond this many combined sigma
A_PLAUSIBLE_MS2 = 200.0          # |a| above this is worth questioning for a coronal shock
REPORT_EXCITER_ENERGY = False    # (gamma-1) m_e c^2 at the shock speed is ~1 eV, far below the
                                 # coronal thermal energy, and is not a property of the emitting
                                 # electrons. Off by default: it can only mislead in a table
INFLATE_BY_CHI2 = False          # if True, widen the height-time bands by sqrt(chi2_red) when the
                                 # fit is worse than the errors imply. OFF by default: for
                                 # comparing models you want to SEE that a band fails to cover the
                                 # data, not have it inflated until it does. chi2_red is reported
                                 # separately either way
FIT_IN_LOGF = True               # fit log10 f(t) rather than f(t). A type II lane spans close to
                                 # an octave here, over which a quadratic in f is far too stiff -
                                 # the drift is much nearer log-linear. Set False for the plain fit
KIN_DEG   = 2                    # degree of the r(t) fit the speed and acceleration come from.
                                 # 2 by default: a cubic can bend a height track into an S and
                                 # then reports a spurious 1000 m/s^2 near the ends
N_BOOT    = 300                  # bootstrap refits for the height-time error bands
N_GRID    = 60                   # points on the shared time grid

# --- Stokes V/I sampling along the lanes ----------------------------------------
POL_DT_S   = 2.0                 # +/- seconds of the box the V/I median is taken over
POL_DF_MHZ = 0.5                 # +/- MHz of that box

JOINT = 'F+H joint'              # key of the combined fundamental + harmonic track


def save_fig(fig, name, dpi=300):
    path = os.path.join(OUTDIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('saved', path)


print(f'event {EVENT_DATE}')
print('bands offered: ' + ', '.join(f'{b} = {BAND_NAME[b]} (s={HARM[b]})' for b in BANDS))
print(f'{N_REPS} repeat(s) per lane')
print('outputs ->', OUTDIR)

## A.1 &middot; The tracing layer and its display stretch

`dyspec_subtracted` computed above is already the right quantity: `df_int` in dB minus the
time-median of every frequency channel, which removes the instrumental bandpass and the quiet-Sun
level and leaves the burst as an excess in dB. It is reused directly rather than recomputed.
Setting `LAYER_MODE = 'ratio'` instead undoes the dB conversion and forms the linear ratio to that
background, displayed on a log colour scale.

The layer is **block-averaged in time** down to at most `LAYER_MAX_T` samples, in chunks so the
full-resolution frame is never copied whole. At the exported 20.97&nbsp;ms cadence the type II
window alone holds of order $10^8$ samples, which no interactive figure can redraw at a usable
rate. `LAYER_MAX_F` is deliberately larger than the 640 channels of this export, so **nothing is
averaged in frequency** &mdash; the band splitting and the fine structure are measured there.

**On the colour stretch.** Two things control how much faint detail survives. `LAYER_PLO` sets the
bottom of the colour scale as a percentile: anything below it is clipped flat, so a high value
erases the background and with it the fine structure. `LAYER_GAMMA` applies a power-law stretch
($\gamma<1$ brightens the faint end without discarding it), which is the gentler instrument for
this. The cell below shows several combinations side by side so you can pick one; the defaults are
`LAYER_PLO = 2`, `LAYER_PHI = 98`, `LAYER_GAMMA = 0.6`.

In [ ]:
def decimate(df, max_t=LAYER_MAX_T, max_f=LAYER_MAX_F, chunk=20000):
    """Block-average a (time x frequency) frame down to at most max_t x max_f samples, using the
    block centres as the new time and frequency coordinates. Averaged in time chunks so the
    full-resolution frame is never copied whole - the type II window holds ~1e8 samples at the
    native NenuFAR cadence. Returns the frame unchanged when it is already small enough."""
    nt, nf = df.shape
    kt = max(1, int(np.ceil(nt / max_t)))
    kf = max(1, int(np.ceil(nf / max_f)))
    if kt == 1 and kf == 1:
        return df, (1, 1)
    nt2, nf2 = (nt // kt) * kt, (nf // kf) * kf
    rows = max(kt, (chunk // kt) * kt)
    blocks = []
    for i0 in tqdm(range(0, nt2, rows), desc='decimating', leave=False):
        i1 = min(i0 + rows, nt2)
        v = df.iloc[i0:i1, :nf2].to_numpy(float)
        blocks.append(np.nanmean(v.reshape((i1 - i0) // kt, kt, nf2 // kf, kf), axis=(1, 3)))
    ti = df.index[:nt2].to_numpy().astype('int64').reshape(-1, kt).mean(axis=1)
    fi = np.asarray(df.columns, float)[:nf2].reshape(-1, kf).mean(axis=1)
    return pd.DataFrame(np.vstack(blocks), index=pd.to_datetime(ti.astype('int64')),
                        columns=fi), (kt, kf)


def window(df):
    """Slice a (time x frequency) frame to the type II window and frequency limits."""
    f = np.asarray(df.columns, float)
    keep = (f >= TYPEII_FLIM[0]) & (f <= TYPEII_FLIM[1])
    return df.loc[max(TYPEII_WINDOW[0], df.index[0]):min(TYPEII_WINDOW[1], df.index[-1]),
                  df.columns[keep]]


# Stokes I tracing layer, straight from the background-subtracted frame computed above
if LAYER_MODE == 'db_sub':
    src_I = dyspec_subtracted
else:
    lin = 10 ** (df_int / 10)
    src_I = lin / np.nanmedian(lin.to_numpy(), axis=0)

LAYER, (kt_I, kf_I) = decimate(window(src_I))
POL, (kt_P, kf_P) = decimate(window(df_pol))
LAYER_T = LAYER.index
LAYER_F = np.asarray(LAYER.columns, float)
LAYER_D = LAYER.to_numpy().T                       # (nfreq, ntime)

print(f'Stokes I layer: {LAYER.shape[0]} time x {LAYER.shape[1]} freq '
      f'(decimated {kt_I}x in time, {kf_I}x in frequency)')
print(f'  {LAYER_T[0].time()}-{LAYER_T[-1].time()} UT, {LAYER_F.min():.1f}-{LAYER_F.max():.1f} MHz,'
      f' cadence {np.nanmedian(np.diff(LAYER_T) / np.timedelta64(1, "ms")):.0f} ms,'
      f' channel {np.nanmedian(np.diff(LAYER_F)) * 1e3:.0f} kHz')
print(f'Stokes V/I layer: {POL.shape[0]} time x {POL.shape[1]} freq '
      f'(decimated {kt_P}x in time, {kf_P}x in frequency)')

In [ ]:
def layer_norm(D, plo=None, phi=None, gamma=None):
    """Colour normalisation for the tracing layer. A low `plo` keeps the faint background instead
    of clipping it flat, and gamma < 1 stretches the faint end up; the two together are what
    decide how much fine structure is visible."""
    plo = LAYER_PLO if plo is None else plo
    phi = LAYER_PHI if phi is None else phi
    gamma = LAYER_GAMMA if gamma is None else gamma
    vmin, vmax = np.nanpercentile(D, plo), np.nanpercentile(D, phi)
    if LAYER_MODE == 'ratio':
        return LogNorm(vmin=max(vmin, 1e-3), vmax=vmax)
    if gamma != 1:
        return PowerNorm(gamma, vmin=vmin, vmax=vmax)
    return Normalize(vmin=vmin, vmax=vmax)


LAST_STRETCH = {}


def draw_layer(ax, plo=2, phi=None, gamma=1, cmap=LAYER_CMAP, tstep=None):
    """Draw the whole tracing layer with pcolormesh in physical (UT, MHz) axes, down-sampled in
    time only for display speed. This is the one figure everything is traced on.

    plo and gamma default to 2 and 1: keep the faint end of the distribution, apply no power-law
    stretch. Pass None for either to fall back to LAYER_PLO / LAYER_GAMMA from the configuration
    cell, or pass explicit values (the stretch-comparison figure does exactly that)."""
    step = max(1, LAYER_D.shape[1] // PREVIEW_MAX_TCOLS) if tstep is None else tstep
    LAST_STRETCH.update(plo=(LAYER_PLO if plo is None else plo),
                        phi=(LAYER_PHI if phi is None else phi),
                        gamma=(LAYER_GAMMA if gamma is None else gamma))
    pm = ax.pcolormesh(LAYER_T[::step], LAYER_F, LAYER_D[:, ::step],
                       norm=layer_norm(LAYER_D, plo, phi, gamma), cmap=cmap, rasterized=True)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    if INVERT_FREQ:
        ax.invert_yaxis()
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=5))
    ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
    ax.set_ylabel('Frequency (MHz)')
    return pm


CBAR_LABEL = 'dB above background' if LAYER_MODE == 'db_sub' else 'ratio to background'

# --- compare a few stretches: pick one and set LAYER_PLO / LAYER_PHI / LAYER_GAMMA above ---
TRIALS = [(60, 99.5, 1.0, 'plo=60, gamma=1  (harsh: faint detail clipped)'),
          (2, 98, 1.0, 'plo=2, gamma=1'),
          (2, 98, 0.6, 'plo=2, gamma=0.6  (default)'),
          (1, 99, 0.4, 'plo=1, gamma=0.4  (softest)')]
fig = plt.figure(figsize=[15, 12])
for i, (plo, phi, gam, ttl) in enumerate(TRIALS, start=1):
    ax = fig.add_subplot(len(TRIALS), 1, i)
    pm = draw_layer(ax, plo=plo, phi=phi, gamma=gam)
    fig.colorbar(pm, ax=ax, pad=0.01, label=CBAR_LABEL)
    ax.set_title(ttl, fontsize=10)
    if i < len(TRIALS):
        ax.set_xlabel('')
fig.suptitle('Choosing the display stretch', y=1.005, fontsize=13)
fig.tight_layout()
save_fig(fig, 'display_stretch_comparison')
plt.show()

> **Figure 1 &mdash; Choosing the display stretch.** `display_stretch_comparison.png`
>
> The same tracing layer rendered with four colour scalings, from the harshest at the top to the
> softest at the bottom. `plo` is the percentile placed at the bottom of the colour bar, so
> everything below it is clipped flat; `gamma` is a power-law stretch applied after normalisation
> ($\gamma<1$ brightens the faint end without discarding it). Only the display changes &mdash; no
> panel here alters a single stored value.
>
> *Why it is here:* how much fine structure you can see is how much you can trace, and a high
> `plo` silently erases the faint band-split branches. Pick a row and set `LAYER_PLO`, `LAYER_PHI`
> and `LAYER_GAMMA` accordingly before tracing.

In [ ]:
fig = plt.figure(figsize=[15, 5])
ax = fig.add_subplot(111)
pm = draw_layer(ax)
fig.colorbar(pm, ax=ax, pad=0.01, label=CBAR_LABEL)
ax.set_title(f'NenuFAR type II tracing layer  (plo={LAST_STRETCH["plo"]}, '
             f'phi={LAST_STRETCH["phi"]}, gamma={LAST_STRETCH["gamma"]})')
fig.tight_layout()
save_fig(fig, 'nenufar_typeii_window')
plt.show()

> **Figure 2 &mdash; The tracing layer.** `nenufar_typeii_window.png`
>
> The NenuFAR Stokes&nbsp;I dynamic spectrum over the type II window, as `dyspec_subtracted`
> (each frequency channel minus its own time median, so the instrumental bandpass and the quiet-Sun
> level are removed and the burst appears as an excess in dB). Frequency increases downwards, as in
> the overview panels above. This is the single image every lane is traced on.
>
> *How it was made:* the frame is sliced to `TYPEII_WINDOW` and `TYPEII_FLIM`, then block-averaged
> **in time only** to at most `LAYER_MAX_T` samples so one interactive figure stays responsive.
> Nothing is averaged in frequency &mdash; the band splitting and the fine structure live there.

## A.2 &middot; Electron-density models &times; fold numbers

Each model returns $n_e(r)$ in cm$^{-3}$ for heliocentric distance $r$ in solar radii. The **fold**
multiplies the whole profile (fold 1 = quiet Sun; 2&ndash;4 brackets a streamer or an active-region
corona), so the model&times;fold grid is the systematic uncertainty on every height-dependent
quantity. A frequency only has a height solution where the implied plasma density falls inside the
model's range; outside it the height is returned as NaN rather than extrapolated.

The panel on the right shows why the two bands are seen at different times. The emission at
$f=s\,f_{pe}$ leaves the observing window as soon as $s\,f_{pe}$ falls outside it, so for a shock
climbing through the corona the fundamental sweeps out through the bottom of the NenuFAR band while
the harmonic is still entering through the top &mdash; the same source, twice, offset in time.

The Mann et&nbsp;al. (2023) constant is 11.14, taken from the text of that paper: it reproduces their
quoted $n_e(3\,R_\odot)=4.267\times10^5$ cm$^{-3}$, whereas the printed Eq.&nbsp;18 shows 11.35.

In [ ]:
def newkirk(r, fold=1):
    """Newkirk (1961), exponential quiet-corona model."""
    r = np.asarray(r, float)
    return fold * 4.2e4 * 10.0 ** (4.32 / r)

def saito(r, fold=1):
    """Saito (1970), equatorial two-term fit."""
    r = np.asarray(r, float)
    return fold * (1.36e6 * r ** -2.14 + 1.68e8 * r ** -6.13)

def leblanc(r, fold=1):
    """Leblanc, Dulk & Bougeret (1998), low corona out to 1 AU."""
    r = np.asarray(r, float)
    return fold * (3.3e5 * r ** -2 + 4.1e6 * r ** -4 + 8.0e7 * r ** -6)

def baumbach_allen(r, fold=1):
    """Baumbach (1937) eclipse fit with the Allen (1947) correction, three-term."""
    r = np.asarray(r, float)
    return fold * 1e8 * (0.036 * r ** -1.5 + 1.55 * r ** -6 + 2.99 * r ** -16)

def mann2023(r, fold=1):
    """Mann, Warmuth, Vocks & Rouillard (2023), A&A 679, A64, Eq. 18 (barometric, PSP-calibrated,
    valid to about 3 Rsun). The constant 11.14 comes from their text and reproduces their quoted
    n_e(3 Rsun) = 4.267e5 cm^-3; the printed equation shows 11.35."""
    r = np.asarray(r, float)
    return fold * 7.17e8 * np.exp(11.14 * (1.0 / r - 1.0))


BASE_MODELS = {'Newkirk': newkirk, 'Saito': saito, 'Leblanc': leblanc,
               'Baumbach-Allen': baumbach_allen, 'Mann 2023': mann2023}

MODEL_GRID = {f'{name} x{fold}': (lambda r, _f=fun, _k=fold: _f(r, fold=_k))
              for name, fun in BASE_MODELS.items() for fold in FOLDS}
print(f'{len(BASE_MODELS)} models x {len(FOLDS)} folds = {len(MODEL_GRID)} combinations')


def freq_to_density(f_hz, harmonic=1):
    """Plasma frequency -> local electron density [cm^-3] for emission at harmonic s. Because the
    harmonic is emitted at 2 f_pe, dividing by s recovers the same local density from either band -
    which is what lets the two bands be joined into one height-time track."""
    return (np.asarray(f_hz, float) / harmonic / PLASMA_CONST) ** 2

def freq_to_radius(f_hz, model, harmonic=1, r_bounds=R_BOUNDS, nres=6000):
    """Invert a monotonically decreasing n_e(r) for the source height; out-of-range -> NaN."""
    rr = np.linspace(r_bounds[0], r_bounds[1], nres)
    ne = model(rr)
    ne_t = freq_to_density(np.atleast_1d(f_hz), harmonic=harmonic)
    r = np.interp(ne_t, ne[::-1], rr[::-1])
    r[(ne_t > np.nanmax(ne)) | (ne_t < np.nanmin(ne))] = np.nan
    return r if r.size > 1 else float(r[0])

def alfven_mach_from_X(X, gamma=5/3):
    """Perpendicular-shock Alfven Mach number from the density jump X (Rankine-Hugoniot),
    valid for 1 <= X < 4. gamma = 5/3 is already baked into the numerical coefficients of this
    standard form, so the argument is kept only to make that assumption explicit."""
    X = np.asarray(X, float)
    out = np.full(X.shape, np.nan)
    ok = (X >= 1) & (X < 4)
    out[ok] = np.sqrt(X[ok] * (X[ok] + 5) / (2 * (4 - X[ok])))
    return out

def electron_energy_from_speed(v_kms):
    """Kinetic energy [keV] of an electron moving at v: E = (gamma - 1) m_e c^2. At a type II
    shock speed this is the bulk exciter energy and is firmly non-relativistic; the relativistic
    form is kept so the same estimator also serves fast type III beams."""
    v = np.asarray(v_kms, float) * 1e3
    beta = np.clip(v / C_MS, 0, 0.999999)
    gamma = 1.0 / np.sqrt(1.0 - beta ** 2)
    return (gamma - 1.0) * M_E * C_MS ** 2 / E_CHARGE_J / 1e3

In [ ]:
rr = np.linspace(1.0, 3.0, 400)
fig = plt.figure(figsize=[14, 5])

ax = fig.add_subplot(121)
for name, fun in BASE_MODELS.items():
    ax.plot(rr, fun(rr), lw=1.8, label=name)
ax.set_yscale('log')
ax.set_xlabel(r'$r\,/\,R_\odot$')
ax.set_ylabel(r'$n_e$ [cm$^{-3}$]')
ax.set_title('(a) Electron-density models (fold 1)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

ax = fig.add_subplot(122)
for b in BANDS:
    for name, fun in BASE_MODELS.items():
        ax.plot(rr, HARM[b] * PLASMA_CONST * np.sqrt(fun(rr)) / 1e6, lw=1.6,
                ls=('-' if b == 'F' else '--'), label=(name if b == 'F' else None))
ax.axhspan(TYPEII_FLIM[0], TYPEII_FLIM[1], color='0.8', alpha=0.5, zorder=0)
ax.text(2.85, np.sqrt(TYPEII_FLIM[0] * TYPEII_FLIM[1]), 'NenuFAR band', ha='right', fontsize=9)
ax.set_yscale('log')
ax.set_ylim(8, 400)
ax.set_xlabel(r'$r\,/\,R_\odot$')
ax.set_ylabel(r'$s\,f_{pe}$ [MHz]')
ax.set_title(r'(b) Emission frequency vs height: $s=1$ solid, $s=2$ dashed')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

fig.tight_layout()
save_fig(fig, 'density_models')
plt.show()

> **Figure 3 &mdash; Coronal electron-density models.** `density_models.png`
>
> *(a)* The five analytic $n_e(r)$ profiles at fold&nbsp;1. *(b)* The emission frequency
> $s\,f_{pe}(r)$ each of them predicts, solid for the fundamental ($s=1$) and dashed for the
> harmonic ($s=2$), with the observing band shaded.
>
> *What to take from it:* panel (b) is the reason the two bands are seen at different times. As the
> shock climbs, the fundamental sweeps down out of the bottom of the band while the harmonic, at
> twice the frequency, is still entering through the top &mdash; the same source, twice, offset in
> time. The vertical spread between the curves at a fixed frequency is the height systematic that
> §A.8 quantifies.

## A.3 &middot; Trace the lanes

**One figure, two ways of laying down a lane.** The whole dynamic spectrum is drawn once and
nothing is split or windowed. A **Method** toggle switches between:

- **Click points** &mdash; left-click along the lane, one point at a time. Best for fine structure
  and for lanes that wander.
- **B&eacute;zier curve** &mdash; place a quadratic or cubic B&eacute;zier on the lane with sliders
  (or type an exact index in the box beside each). The curve updates live and is sampled at
  `BEZIER_NUM_POINTS` points when you record it. Best for the smooth drifting envelope of a band,
  and much quicker than clicking a long lane.

Either way you press **End trace** to record what you have placed, under the current label.

| control | what it does |
|---|---|
| **Band** | whether the next trace is a fundamental or a harmonic lane |
| **Method** | click points, or place a B&eacute;zier curve |
| **End trace** (&#10003;) | record the trace in progress under the current label. In click mode a right-click does the same; in B&eacute;zier mode clicks on the plot are ignored, so this button is the only way to commit |
| **Undo point** (&#8630;) | remove the last clicked point (click mode only) |
| **New lane** (+) | move on to the next free lane number of this band |
| **Delete last** | remove the most recently recorded repeat |
| **auto-repeats** | B&eacute;zier mode only: record all `N_REPS` repeats at once, the first being the curve you placed and the rest jittered by `BEZIER_JITTER` pixels |

Traces are labelled **`F lane 1`, `F lane 2`, `H lane 1`** and so on. Each label is traced
`N_REPS` times &mdash; the spread between repeats is the reproducibility error &mdash; and the
tracer moves to the next lane automatically once a label has its repeats. Trace as many lanes per
band as the burst actually shows; you do not have to say which is the lower or upper branch of a
split, because &sect;A.4 sorts the lanes of each band in frequency and takes the lowest as the
upstream branch and the next as the downstream one.

**Your tracing is safe against re-runs.** The recorded traces live in a module-level `TRACE_STORE`,
not on the tracer object, so re-running the cell that opens the tracer picks up exactly what you
had (it prints what it restored). Call `LaneTracer(reset=True)` when you actually want to start
over. The lane number is derived from what is recorded rather than from a counter, so deleting
traces walks the numbering back down instead of leaving it stranded above the real lanes. Run
`tracer.summary()` at any point to see precisely what is stored.

The figure needs the widget backend (`%matplotlib widget`, i.e. `ipympl`). Zoom and pan with the
toolbar as usual; clicks are ignored while a toolbar tool is active, so you can zoom in on the fine
structure and then trace it.

In [ ]:
def draw_bezier(x1=0, y1=0, x2=0, y2=0, controls=[[0, 0]], n=2, num_points=30):
    """Bezier curve of degree n from (x1,y1) to (x2,y2) with n-1 control points.
    n=1 linear, n=2 quadratic (1 anchor), n=3 cubic (2 anchors). Returns (num_points, 2)."""
    P0 = np.array([x1, y1], dtype=float)
    P3 = np.array([x2, y2], dtype=float)
    t = np.linspace(0, 1, num_points)
    if n == 1:
        curve = (1 - t)[:, None] * P0 + t[:, None] * P3
    elif n == 2:
        P1 = np.array(controls[0], dtype=float)
        curve = ((1 - t)[:, None] ** 2 * P0
                 + 2 * (1 - t)[:, None] * t[:, None] * P1
                 + t[:, None] ** 2 * P3)
    elif n == 3:
        P1 = np.array(controls[0], dtype=float)
        P2 = np.array(controls[1], dtype=float)
        curve = ((1 - t)[:, None] ** 3 * P0
                 + 3 * (1 - t)[:, None] ** 2 * t[:, None] * P1
                 + 3 * (1 - t)[:, None] * t[:, None] ** 2 * P2
                 + t[:, None] ** 3 * P3)
    else:
        raise ValueError('n must be 1 (linear), 2 (quadratic) or 3 (cubic).')
    return curve


def bezier_indices(x1, y1, x2, y2, controls, n, num_points=BEZIER_NUM_POINTS):
    """Integer (time-index, frequency-index) samples along a Bezier placed in the index space of
    the tracing layer, clipped to its bounds."""
    curve = draw_bezier(x1, y1, x2, y2, controls, n, num_points)
    xi = np.clip(np.round(curve[:, 0]).astype(int), 0, len(LAYER_T) - 1)
    yi = np.clip(np.round(curve[:, 1]).astype(int), 0, len(LAYER_F) - 1)
    return xi, yi


def bezier_freq_lane(x_idx, y_idx):
    """Map integer (time-idx, freq-idx) curve samples onto a {'t': [...], 'f': [...]} lane in MHz,
    sorted in time with duplicate times collapsed so the polynomial fits stay well posed."""
    t_sel = pd.to_datetime([LAYER_T[i] for i in x_idx])
    f_val = LAYER_F[y_idx]
    x_num = mdates.date2num(t_sel)
    order = np.argsort(x_num)
    t_sorted = np.asarray(t_sel)[order]
    f_sorted = f_val[order]
    _, uniq = np.unique(x_num[order], return_index=True)
    return {'t': list(t_sorted[uniq]), 'f': list(f_sorted[uniq])}

In [ ]:
# Recorded traces live in a module-level store, NOT on the tracer object, so re-running this
# cell or the one that opens the tracer never throws your tracing away. Pass reset=True to
# LaneTracer when you really do want to start from scratch.
TRACE_STORE = {}         # label -> list of repeats, each {'t': [...], 'f': [...]}
TRACE_HISTORY = []       # labels in the order they were actually recorded, for Delete last
TRACE_KIND = {}          # label -> how its repeats were produced. This matters: jittered copies
                         # of one Bezier are NOT independent re-tracings and their spread is not a
                         # reproducibility error


class LaneTracer:
    """Lane tracer on a single plot of the whole dynamic spectrum, with two interchangeable
    methods: clicking points, or placing a Bezier curve with sliders. Both record under the same
    labels ('F lane 1', ...) and feed the identical `passes` structure. The spectrogram is drawn
    once and only the overlays are redrawn, through blitting.

    The lane number is always derived from what is actually recorded rather than from a counter,
    so deleting traces walks the numbering back down instead of leaving it stranded above the
    real lanes."""

    SLIDERS = [('x_start', 'start x (time)', 'x'), ('y_start', 'start y (freq)', 'y'),
               ('x_end', 'end x (time)', 'x'), ('y_end', 'end y (freq)', 'y'),
               ('cx1', 'anchor 1 x', 'x'), ('cy1', 'anchor 1 y', 'y'),
               ('cx2', 'anchor 2 x', 'x'), ('cy2', 'anchor 2 y', 'y')]

    def __init__(self, n_reps=N_REPS, bands=None, method=TRACE_METHOD, anchors=BEZIER_ANCHORS,
                 reset=False):
        import ipywidgets as widgets
        from IPython.display import display
        if reset:
            TRACE_STORE.clear()
            TRACE_HISTORY.clear()
            TRACE_KIND.clear()
        self.n_reps = n_reps
        self.bands = list(bands) if bands else list(BANDS)
        self.band = self.bands[0]
        self.method = method
        self.n_anchors = anchors
        self.traces = TRACE_STORE            # bound, not copied: survives re-running the cell
        self.history = TRACE_HISTORY
        self.lane_no = {b: 1 for b in self.bands}
        self.current = {'t': [], 'f': []}
        self.rng = np.random.default_rng(BEZIER_SEED)
        for b in self.bands:
            self._sync_label(b)
        if self.traces:
            print(f'restored {len(self.traces)} lane(s) already recorded: '
                  + ', '.join(f'{l} ({len(v)} rep)' for l, v in sorted(self.traces.items())))
            print('pass reset=True to LaneTracer if you want to start over')
        self._build(widgets, display)

    # ---------------------------------------------------------------- labels and state
    @property
    def label(self):
        return f'{self.band} lane {self.lane_no[self.band]}'

    def _used(self, band):
        return sorted(int(l.split()[-1]) for l in self.traces if l.startswith(band + ' lane '))

    def _sync_label(self, band=None):
        """Point the current label at a lane that is still being filled, otherwise at the next
        free number. Derived from `self.traces`, never from a free-running counter."""
        b = band or self.band
        used = self._used(b)
        n_cur = len(self.traces.get(f'{b} lane {self.lane_no[b]}', []))
        if 0 < n_cur < self.n_reps:
            return
        self.lane_no[b] = (max(used) + 1) if used else 1

    @property
    def labels(self):
        """Every recorded label, ordered by band then lane number."""
        return sorted(self.traces, key=lambda l: (self.bands.index(l.split()[0]),
                                                  int(l.split()[-1])))

    @property
    def order(self):
        return self.labels

    def colour(self, label):
        b = label.split()[0]
        i = int(label.split()[-1]) - 1
        cmap = plt.cm.winter if b == 'F' else plt.cm.autumn
        return cmap(min(i, 3) / 3.5)

    @property
    def passes(self):
        """The k-th repeat of every label assembled into pass k, so the lanes of a pass are
        mutually consistent. Labels with fewer repeats cycle through the ones they have."""
        if not self.traces:
            return []
        n = max(len(v) for v in self.traces.values())
        out = []
        for k in range(n):
            pas = {}
            for lab in self.labels:
                reps = self.traces[lab]
                if reps:
                    r = reps[k % len(reps)]
                    pas[lab] = {'t': list(r['t']), 'f': list(r['f'])}
            out.append(pas)
        return out

    def summary(self):
        """What is currently recorded. Run `tracer.summary()` at any point to check."""
        rows = []
        for lab in self.labels:
            reps = self.traces[lab]
            f = np.concatenate([np.asarray(r['f'], float) for r in reps])
            t = pd.to_datetime(np.concatenate([np.asarray(r['t']) for r in reps]))
            rows.append({'lane': lab, 'band': lab.split()[0], 'repeats': len(reps),
                         'repeat_kind': TRACE_KIND.get(lab, 'independent traces'),
                         'points_per_repeat': int(np.median([len(r['f']) for r in reps])),
                         'start_UT': t.min().strftime('%H:%M:%S'),
                         'end_UT': t.max().strftime('%H:%M:%S'),
                         'f_min_MHz': round(float(f.min()), 2),
                         'f_max_MHz': round(float(f.max()), 2)})
        return pd.DataFrame(rows)

    # ---------------------------------------------------------------- figure and widgets
    def _build(self, widgets, display):
        from ipywidgets import IntSlider, BoundedIntText, HBox, VBox, Label, Layout, jslink
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=[13, 6])
        plt.ion()
        try:
            self.fig.canvas.header_visible = False
        except AttributeError:
            pass
        pm = draw_layer(self.ax)
        self.fig.colorbar(pm, ax=self.ax, pad=0.01, label=CBAR_LABEL)

        (self.live,) = self.ax.plot([], [], 'o-', color='k', ms=6, lw=1.4, mfc='white',
                                    mew=1.4, animated=True)
        (self.bez_line,) = self.ax.plot([], [], '-', color='k', lw=2.2, animated=True)
        (self.bez_ends,) = self.ax.plot([], [], 'o', mfc='white', mec='black', mew=1.5, ms=9,
                                        ls='none', animated=True)
        (self.bez_anch,) = self.ax.plot([], [], '^', mfc='white', mec='black', mew=1.3, ms=10,
                                        ls='none', animated=True)
        self.done_lines = {}
        self._bg = None
        self.fig.canvas.mpl_connect('draw_event', self._on_draw)
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)

        self.sel = widgets.ToggleButtons(
            options=[(f'{BAND_NAME[b]} ({b})', b) for b in self.bands], value=self.band,
            description='Band:', style={'description_width': 'initial'})
        self.meth = widgets.ToggleButtons(
            options=[('Click points', 'click'), ('Bezier curve', 'bezier')], value=self.method,
            description='Method:', style={'description_width': 'initial'})
        self.deg = widgets.Dropdown(options=[('quadratic', 1), ('cubic', 2)], value=self.n_anchors,
                                    description='Bezier:', layout=Layout(width='170px'),
                                    style={'description_width': 'initial'})
        self.jit = widgets.Checkbox(value=True, description=f'auto-repeats ({self.n_reps})',
                                    indent=False, layout=Layout(width='170px'))
        self.btn_end = widgets.Button(description='End trace', icon='check',
                                      button_style='success',
                                      tooltip='record the trace in progress')
        self.btn_undo = widgets.Button(description='Undo point', icon='rotate-left')
        self.btn_new = widgets.Button(description='New lane', icon='plus',
                                      tooltip='start the next lane of this band')
        self.btn_del = widgets.Button(description='Delete last', icon='trash',
                                      button_style='danger',
                                      tooltip='drop the most recently recorded trace')
        self.status = widgets.HTML()

        d = self._defaults()
        self._sliders, rows = {}, []
        for name, lab, axis in self.SLIDERS:
            hi = (len(LAYER_T) if axis == 'x' else len(LAYER_F)) - 1
            sld = IntSlider(value=d[name], min=0, max=hi, step=1, readout=False,
                            continuous_update=True, layout=Layout(width='520px'))
            box = BoundedIntText(value=d[name], min=0, max=hi, step=1, layout=Layout(width='95px'))
            jslink((sld, 'value'), (box, 'value'))
            sld.observe(self._on_slider, names='value')
            self._sliders[name] = sld
            rows.append(HBox([Label(lab, layout=Layout(width='120px')), sld, box]))
        self._rows = rows
        self.bez_box = VBox(rows)

        self.sel.observe(self._on_band, names='value')
        self.meth.observe(self._on_method, names='value')
        self.deg.observe(self._on_deg, names='value')
        self.btn_end.on_click(lambda _: self.end_trace())
        self.btn_undo.on_click(lambda _: self.undo())
        self.btn_new.on_click(lambda _: self.new_lane())
        self.btn_del.on_click(lambda _: self.delete_last())
        display(VBox([HBox([self.sel, self.meth]),
                      HBox([self.btn_end, self.btn_undo, self.btn_new, self.btn_del,
                            self.deg, self.jit]),
                      self.bez_box, self.status, self.fig.canvas]))
        for lab in self.labels:
            self._redraw_label(lab)
        self._apply_method()
        self._title()
        self._refresh()

    def _defaults(self):
        """Seed the Bezier on the brightest channel near the start, middle and end of the window,
        so the curve already lies near an emission lane before you touch it."""
        nt, nf = len(LAYER_T), len(LAYER_F)
        xs = [int(f * (nt - 1)) for f in (0.10, 0.35, 0.65, 0.90)]
        w = max(1, nt // 40)
        ys = []
        for x in xs:
            col = np.nanmean(LAYER_D[:, max(0, x - w):min(nt, x + w) + 1], axis=1)
            ys.append(int(np.nanargmax(col)) if np.isfinite(col).any() else nf // 2)
        return dict(x_start=xs[0], y_start=ys[0], x_end=xs[3], y_end=ys[3],
                    cx1=xs[1], cy1=ys[1], cx2=xs[2], cy2=ys[2])

    def _title(self):
        n = len(self.traces.get(self.label, []))
        how = 'click points' if self.method == 'click' else 'place the Bezier with the sliders'
        self.ax.set_title(f'tracing {self.label}  -  repeat {min(n + 1, self.n_reps)} of '
                          f'{self.n_reps}   |   {how}, then End trace')

    # ---------------------------------------------------------------- blitting
    def _on_draw(self, event=None):
        self._bg = self.fig.canvas.copy_from_bbox(self.ax.bbox)
        self._blit()

    def _blit(self):
        for ln in self.done_lines.values():
            self.ax.draw_artist(ln)
        for art in (self.live, self.bez_line, self.bez_ends, self.bez_anch):
            self.ax.draw_artist(art)

    def _refresh(self):
        if self.method == 'click':
            if self.current['f']:
                self.live.set_data(mdates.date2num(pd.to_datetime(self.current['t'])),
                                   self.current['f'])
            else:
                self.live.set_data([], [])
        else:
            v = {k: s.value for k, s in self._sliders.items()}
            anch = self._anchors(v)
            curve = draw_bezier(v['x_start'], v['y_start'], v['x_end'], v['y_end'],
                                anch, self.n_anchors + 1, BEZIER_NUM_POINTS)
            xi = np.clip(np.round(curve[:, 0]).astype(int), 0, len(LAYER_T) - 1)
            yi = np.clip(np.round(curve[:, 1]).astype(int), 0, len(LAYER_F) - 1)
            self.bez_line.set_data(mdates.date2num(pd.to_datetime(LAYER_T[xi])), LAYER_F[yi])
            self.bez_ends.set_data(mdates.date2num(pd.to_datetime(
                LAYER_T[[v['x_start'], v['x_end']]])), LAYER_F[[v['y_start'], v['y_end']]])
            self.bez_anch.set_data(mdates.date2num(pd.to_datetime(
                LAYER_T[[int(a[0]) for a in anch]])), LAYER_F[[int(a[1]) for a in anch]])
        if self._bg is not None:
            self.fig.canvas.restore_region(self._bg)
            self._blit()
            self.fig.canvas.blit(self.ax.bbox)
        else:
            self.fig.canvas.draw_idle()
        self._update_status()

    def _line_for(self, label):
        if label not in self.done_lines:
            (self.done_lines[label],) = self.ax.plot([], [], '-', lw=2.0, alpha=0.9,
                                                     color=self.colour(label), animated=True,
                                                     label=label)
        return self.done_lines[label]

    def _redraw_label(self, label):
        xs, ys = [], []
        for rep in self.traces.get(label, []):
            xs += list(mdates.date2num(pd.to_datetime(rep['t']))) + [np.nan]
            ys += list(rep['f']) + [np.nan]
        self._line_for(label).set_data(xs, ys)

    # ---------------------------------------------------------------- Bezier helpers
    def _anchors(self, v):
        if self.n_anchors == 1:
            return [(v['cx1'], v['cy1'])]
        return [(v['cx1'], v['cy1']), (v['cx2'], v['cy2'])]

    def _bezier_rep(self, jitter=0.0):
        v = {k: s.value for k, s in self._sliders.items()}
        base = np.array([[v['x_start'], v['y_start']], [v['x_end'], v['y_end']],
                         *self._anchors(v)], float)
        if jitter > 0:
            base = base + self.rng.normal(0, jitter, base.shape)
        (sx, sy), (ex, ey) = base[0], base[1]
        xi, yi = bezier_indices(sx, sy, ex, ey, [list(p) for p in base[2:]], self.n_anchors + 1)
        return bezier_freq_lane(xi, yi)

    def _apply_method(self):
        show = '' if self.method == 'bezier' else 'none'
        self.bez_box.layout.display = show
        self.deg.layout.display = show
        self.jit.layout.display = show
        self.btn_undo.disabled = self.method != 'click'
        for i, row in enumerate(self._rows):
            row.layout.display = 'none' if (self.method != 'bezier' or
                                            (i >= 6 and self.n_anchors == 1)) else ''

    # ---------------------------------------------------------------- interaction
    def _on_band(self, change):
        self.band = change['new']
        self.current = {'t': [], 'f': []}
        self._sync_label()
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()

    def _on_method(self, change):
        self.method = change['new']
        self.current = {'t': [], 'f': []}
        self.live.set_data([], [])
        for art in (self.bez_line, self.bez_ends, self.bez_anch):
            art.set_data([], [])
        self._apply_method()
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()

    def _on_deg(self, change):
        self.n_anchors = change['new']
        self._apply_method()
        self._refresh()

    def _on_slider(self, change=None):
        self._refresh()

    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None:
            return
        mode = getattr(getattr(self.fig.canvas, 'toolbar', None), 'mode', '')
        if mode:                                  # zoom or pan is active, so do not add points
            return
        if self.method != 'click':
            # in Bezier mode the curve is placed with the sliders, so a stray click on the plot
            # must not record anything: End trace is the only way to commit it
            return
        if event.button == 3:
            self.end_trace()
            return
        self.current['t'].append(pd.Timestamp(mdates.num2date(event.xdata).replace(tzinfo=None)))
        self.current['f'].append(float(event.ydata))
        self._refresh()

    def undo(self):
        if self.method == 'click' and self.current['t']:
            self.current['t'].pop()
            self.current['f'].pop()
        self._refresh()

    def end_trace(self):
        lab = self.label
        if self.method == 'click':
            if len(self.current['f']) < 2:
                self._update_status('a trace needs at least two points', warn=True)
                return
            new_reps = [{'t': list(self.current['t']), 'f': list(self.current['f'])}]
            self.current = {'t': [], 'f': []}
        else:
            done = len(self.traces.get(lab, []))
            if done >= self.n_reps:
                self._update_status(f'{lab} already has its {self.n_reps} repeats - press '
                                    '"New lane" to move on', warn=True)
                return
            if self.jit.value:
                # repeat 0 is the curve you placed; the rest jitter the control points, which is
                # the Bezier equivalent of re-tracing a lane by hand
                new_reps = [self._bezier_rep(0.0 if (done + k) == 0 else BEZIER_JITTER)
                            for k in range(self.n_reps - done)]
            else:
                new_reps = [self._bezier_rep(0.0)]
        self.traces.setdefault(lab, []).extend(new_reps)
        self.history.extend([lab] * len(new_reps))
        if self.method == 'bezier' and self.jit.value and len(new_reps) > 1:
            TRACE_KIND[lab] = 'bezier auto-repeats'
        else:
            TRACE_KIND.setdefault(lab, 'independent traces')
        self._redraw_label(lab)
        msg = f'recorded {lab}, {len(self.traces[lab])}/{self.n_reps} repeat(s)'
        self._sync_label()
        if self.label != lab:
            msg += f' - moving on to {self.label}'
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()
        self._update_status(msg)

    def new_lane(self):
        used = self._used(self.band)
        self.lane_no[self.band] = (max(used) + 1) if used else 1
        self.current = {'t': [], 'f': []}
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()
        self._update_status(f'started {self.label}')

    def delete_last(self):
        if not self.history:
            self._update_status('nothing recorded yet', warn=True)
            return
        lab = self.history.pop()
        if self.traces.get(lab):
            self.traces[lab].pop()
        if lab in self.traces and not self.traces[lab]:
            del self.traces[lab]
        self._redraw_label(lab)
        self._sync_label(lab.split()[0])
        self._sync_label()
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()
        self._update_status(f'deleted one repeat of {lab}, '
                            f'{len(self.traces.get(lab, []))} left on it')

    def _update_status(self, msg='', warn=False):
        if self.traces:
            bits = []
            for lab in self.labels:
                reps = len(self.traces[lab])
                col = mpl.colors.to_hex(self.colour(lab))
                mark = '' if reps >= self.n_reps else ' <i>(incomplete)</i>'
                bits.append(f'<span style="color:{col}"><b>{lab}</b> {reps}/{self.n_reps}'
                            f'{mark}</span>')
            rec = ' &nbsp;|&nbsp; '.join(bits)
            tot = (f'{len(self.traces)} lane(s), '
                   f'{sum(len(v) for v in self.traces.values())} trace(s)')
        else:
            rec = '<i>nothing recorded yet</i>'
            tot = '0 lanes'
        extra = (f'&nbsp; <b>points in this trace:</b> {len(self.current["f"])}'
                 if self.method == 'click' else '&nbsp; <b>method:</b> Bezier')
        colour = '#a00' if warn else '#060'
        self.status.value = (f'<b>now tracing:</b> {self.label} {extra}<br>'
                             f'<b>recorded ({tot}):</b> {rec}<br>'
                             f'<span style="color:{colour}">{msg}</span>')

Run the cell below to open the tracer. Everything is traced in that one figure, in whichever
method you prefer &mdash; you can switch between clicking and B&eacute;zier at any point, even
between lanes. When you are done, run the cell after it to collect the traces.

In [ ]:
%matplotlib widget
tracer = LaneTracer()          # LaneTracer(reset=True) to discard everything and start over

In [ ]:
%matplotlib inline
# ---- collect the traces -------------------------------------------------------------------
passes = tracer.passes
if not passes:
    raise RuntimeError('nothing traced yet - trace the lanes in the figure above, press '
                       '"End trace" for each, then re-run this cell')

TRACED = [lab for lab in tracer.labels if any(p.get(lab, {}).get('f') for p in passes)]
LANE_COL = {lab: tracer.colour(lab) for lab in TRACED}
LANE_COL[JOINT] = 'black'
LANE_BAND = {lab: lab.split()[0] for lab in TRACED}
BANDS_TRACED = [b for b in BANDS if b in set(LANE_BAND.values())]

# say plainly what came out of the store, and complain if it does not look like a traced event
n_by_band = {b: sum(1 for l in TRACED if l.startswith(b + ' ')) for b in BANDS}
print(f'{len(passes)} pass(es) over {len(TRACED)} lane(s): '
      + ', '.join(f'{n_by_band[b]} x {BAND_NAME[b]}' for b in BANDS))
for b in BANDS:
    if n_by_band[b] == 0:
        print(f'  WARNING: no {BAND_NAME[b]} lane recorded - the {b} band diagnostics will be '
              'skipped. Switch the Band toggle and trace it, then re-run this cell.')
    elif n_by_band[b] == 1:
        print(f'  NOTE: only one {BAND_NAME[b]} lane recorded, so that band has no split, '
              'hence no X or M_A. Trace its other split branch if the spectrum shows one.')
incomplete = [l for l in TRACED if len(tracer.traces[l]) < tracer.n_reps]
if incomplete:
    print(f'  NOTE: fewer than {tracer.n_reps} repeats on: ' + ', '.join(incomplete))
AUTO_REPEAT = [l for l in TRACED if TRACE_KIND.get(l) == 'bezier auto-repeats']
if AUTO_REPEAT:
    print(f'\n  NOTE: {", ".join(AUTO_REPEAT)} used Bezier AUTO-REPEATS. Those repeats are the '
          f'same curve with its\n        control points jittered by BEZIER_JITTER = '
          f'{BEZIER_JITTER} pixels, so their spread measures that jitter,\n        NOT how '
          'reproducibly you can place a curve on the lane. The error bars for those lanes are\n'
          '        therefore fit-error-dominated and optimistic. For a genuine reproducibility '
          'error, untick\n        "auto-repeats" and place the curve independently '
          f'{tracer.n_reps} times.')

tracer.summary()

> **Table 1 &mdash; What is recorded.** `tracer.summary()`
>
> One row per traced lane: how many repeats it has, **how those repeats were produced**
> (`repeat_kind`), how many points each holds, and the time and frequency range it covers.
>
> *Read `repeat_kind` carefully.* `independent traces` means you placed the lane that many times by
> hand, so the spread between them is a genuine reproducibility error. `bezier auto-repeats` means
> repeat&nbsp;0 is the curve you placed and the rest are that same curve with its control points
> jittered by `BEZIER_JITTER` pixels &mdash; their spread measures the jitter, not your ability to
> place a curve, so the error bars for those lanes are optimistic.

In [ ]:
fig = plt.figure(figsize=[15, 6])
ax = fig.add_subplot(111)
pm = draw_layer(ax)
fig.colorbar(pm, ax=ax, pad=0.01, label=CBAR_LABEL)
for lab in TRACED:
    for k, pas in enumerate(passes):
        L = pas.get(lab)
        if L and L['f']:
            ax.plot(pd.to_datetime(L['t']), L['f'], color=LANE_COL[lab],
                    lw=(2.2 if k == 0 else 0.8), alpha=(1.0 if k == 0 else 0.4),
                    label=(lab if k == 0 else None))
ax.set_title('Traced lanes (bold = first repeat, thin = the other repeats)')
ax.legend(fontsize=8, ncol=4, loc='lower left')
fig.tight_layout()
save_fig(fig, 'traced_lanes_preview')
plt.show()

# --- tracing quality: how far is each traced point from the local intensity ridge? ---
# worth checking in Bezier mode especially: a Bezier's interior control points are NOT on its
# curve, so a curve can look plausible while sitting systematically off the lane
qc = []
for lab in TRACED:
    L = passes[0][lab]
    ti = np.searchsorted(LAYER_T, pd.to_datetime(L['t']).to_numpy()).clip(0, len(LAYER_T) - 1)
    fi = np.abs(LAYER_F[:, None] - np.asarray(L['f'], float)[None, :]).argmin(axis=0)
    half = max(3, len(LAYER_F) // 60)
    off = []
    for it, jf in zip(ti, fi):
        lo, hi = max(0, jf - half), min(len(LAYER_F), jf + half + 1)
        col = LAYER_D[lo:hi, it]
        if np.isfinite(col).any():
            off.append(int(np.nanargmax(col)) + lo - jf)
    off = np.asarray(off, float)
    # how bright is the layer ON the trace? A lane drawn across empty spectrum has a small
    # offset-from-ridge by accident (the brightest noise pixel in the window is near zero
    # offset), so the offset check alone cannot tell it from a good trace. The intensity can.
    amp = np.array([LAYER_D[jf, it] for it, jf in zip(ti, fi)], float)
    qc.append({'lane': lab, 'median_offset_chan': np.nanmedian(off),
               'rms_offset_chan': np.sqrt(np.nanmean(off ** 2)),
               'offset_MHz': np.nanmedian(off) * np.nanmedian(np.diff(LAYER_F)),
               'median_on_trace': np.nanmedian(amp), 'min_on_trace': np.nanmin(amp)})
qc = pd.DataFrame(qc)

# the rms offset from the ridge is the honest 1-sigma frequency uncertainty of a traced point:
# it is how far the curve actually sits from the emission it is meant to follow
_chan = float(np.nanmedian(np.diff(LAYER_F)))
if LANE_SIGMA_MHZ == 'auto':
    LANE_SIGMA = {r['lane']: max(abs(r['rms_offset_chan']) * _chan, LANE_SIGMA_FLOOR)
                  for _, r in qc.iterrows()}
elif LANE_SIGMA_MHZ is None:
    LANE_SIGMA = {}
else:
    LANE_SIGMA = {lab: float(LANE_SIGMA_MHZ) for lab in TRACED}
qc['sigma_f_MHz'] = [LANE_SIGMA.get(l, np.nan) for l in qc['lane']]

_unit = 'dB above background' if LAYER_MODE == 'db_sub' else 'x background'
print(f'offset of each traced point from the local intensity maximum '
      f'(search window +/-{half} channels, channel width {_chan * 1e3:.0f} kHz).')
print('more than a channel or two of median offset means the trace is sitting off the lane.')
print(f'median_on_trace / min_on_trace are the layer value ON the trace, in {_unit}.')
print('sigma_f_MHz is the per-point frequency uncertainty the lane fits will use.')
_faint = qc[qc['median_on_trace'] < QC_MIN_SNR_DB]
if len(_faint):
    print()
    for _, r in _faint.iterrows():
        print(f'  *** WARNING: {r["lane"]} sits at only {r["median_on_trace"]:.1f} '
              f'{_unit} along its length (threshold {QC_MIN_SNR_DB}). Part of it is very likely '
              'drawn across empty spectrum - check it against the traced-lanes figure before '
              'using it. A low offset-from-ridge does NOT vouch for a trace over empty sky.')
qc.round(3)

> **Figure 4 &mdash; Traced lanes.** `traced_lanes_preview.png`
>
> Every recorded lane drawn on the tracing layer; the bold line is the first repeat and the thin
> ones are the others, so the visible thickness of a bundle *is* the reproducibility spread.
>
> **Table 2 &mdash; Tracing quality.** (printed below the figure)
>
> For each lane: the median and rms offset, in frequency channels, between the traced points and
> the local intensity maximum within a $\pm$ few-channel search window; the layer value **on** the
> trace; and `sigma_f_MHz`, the per-point frequency uncertainty the lane fits will use.
>
> *Both columns are needed.* A median offset above a channel or two means the curve is sitting
> beside the lane rather than on it. But the offset test alone cannot detect a lane drawn across
> **empty spectrum** &mdash; over blank sky the brightest pixel inside the search window is noise,
> which lands near zero offset by chance. `median_on_trace` is what catches that, and a lane below
> `QC_MIN_SNR_DB` is flagged.

## A.4 &middot; Shock diagnostics from the traced lanes

**Every lane is analysed on its own.** Nothing is merged. `MAKE_JOINT = True` additionally builds
an `F+H joint` track that averages the two bands' *upstream* lanes where they overlap and takes
whichever exists elsewhere; it buys a longer baseline for the second derivative but mixes two
independent measurements, so it is off by default.

**Which lane is which.** The lanes of each band are ordered by frequency **over the interval in
which they overlap in time**: the lower is the **upstream** (unshocked) branch of that band's
split, the next the **downstream** (compressed) branch, so $X=(f_U/f_L)^2\ge1$. Comparing over the
overlap rather than over each lane's own span matters &mdash; a long lane running high to low can
have a higher own-span mean than a short lane sitting above it, and ordering on that inverts the
split, sends $X$ below 1, and makes $M_A$, $v_A$ and $B$ all NaN.

### The estimators, in order

1. **Lane fit.** Each traced lane is fitted with a degree-`POLY_DEG` polynomial in $\log_{10}f$
   against time. Log space because a type II lane spans close to an octave, over which a
   polynomial in $f$ is far too stiff. Drift rates come off that fit analytically:
   $\dot f = f\ln 10\,\mathrm{d}\log_{10}f/\mathrm{d}t$, and the relative drift
   $\dot f/f=\ln 10\,\mathrm{d}\log_{10}f/\mathrm{d}t$ is a property of the shock alone, free of
   the band's harmonic number.
2. **Band split.** $X=(f_U/f_L)^2$ and $M_A=\sqrt{X(X+5)/[2(4-X)]}$, evaluated where both branches
   of a band exist. No density model enters, so each band gives an independent measurement.
3. **Height.** $n_e=(f/s/\mathrm{PLASMA\_CONST})^2$ with $s=1$ for F and $s=2$ for H, then
   $n_e(r)$ is inverted numerically for $r$. Out-of-range densities return NaN rather than an
   extrapolation.
4. **Speed and acceleration.** $r(t)$ is fitted with a degree-`KIN_DEG` polynomial and
   differentiated analytically. `KIN_DEG = 2`, so $a$ is constant per lane; §A.7 gives the
   time-dependent version. The fit is skipped unless the model resolves at least
   `KIN_MIN_FRAC` of that **lane's own** traced span with at least `KIN_MIN_PTS` points &mdash;
   judged against the lane, not the shared grid, so a short lane is not thrown away.
5. **Alfv&eacute;n speed and field.** $v_A=v_{\rm sh}/M_A$ and $B=v_A\sqrt{\mu_0\rho}$ with
   $\rho=\mu m_p n_e$. Both inherit $M_A$ from that lane's band, so a lane whose band has only one
   traced branch has no $v_A$ and no $B$.

### Where the error bars come from, and why they are small

Two sources are propagated together by Monte Carlo. The **repeat error** is the scatter between
your repeats of a lane. The **fit error** is drawn from the covariance of the lane's polynomial
coefficients: for each pass, `N_MC` coefficient vectors are drawn from that covariance and pushed
through the entire chain above. The quoted `_sd` is the standard deviation over all
draws&times;passes and the plotted `_se` is that divided by $\sqrt{N_{\rm pass}}$.

Two things are done to stop that error being fictitiously small. A B&eacute;zier is sampled at
`BEZIER_NUM_POINTS` along a curve with only about eight free parameters, so those samples are
nowhere near independent; the fit **thins them to one point per `FIT_MIN_DT_S` seconds**. And
`np.polyfit`'s default covariance scales with the residual scatter, which for a smooth curve
measures how smooth the curve is rather than how well it sits on the lane; each point is instead
given an explicit frequency uncertainty (`LANE_SIGMA_MHZ = 'auto'` takes the rms offset from the
ridge measured by the §A.3 QC) and the covariance is returned **unscaled**.

Even so the height errors stay small, and that is real rather than a bug: in an exponential corona
$r$ depends only weakly on $f$. For Newkirk at $r\simeq1.8\,R_\odot$,
$\mathrm{d}\ln r/\mathrm{d}\ln f\approx-0.35$, so a 1% frequency error is a 0.35% height error,
about $0.006\,R_\odot$. **The uncertainty that matters is the density model**, and that is the
model&times;fold range in §A.8. Quote the reference-model value with that range beside it.

In [ ]:
t0 = TYPEII_WINDOW[0]        # shared time origin for every lane


def _thin(ts, min_dt=FIT_MIN_DT_S):
    """Keep one point per min_dt seconds. Points sampled densely along a Bezier are not
    independent measurements - the curve has ~8 free parameters - and feeding all of them to
    polyfit shrinks the covariance by a factor it has not earned."""
    keep, last = [], -np.inf
    for i, t in enumerate(ts):
        if t - last >= min_dt:
            keep.append(i)
            last = t
    return np.asarray(keep, int)


def _lane_fit(lane_data, deg=POLY_DEG, sigma_f=None):
    """Polynomial fit of a lane, in log10 f when FIT_IN_LOGF.

    sigma_f is the assumed 1-sigma frequency uncertainty of a traced point in MHz. When given,
    the fit is weighted by it and the covariance is returned UNSCALED, so it reflects that stated
    uncertainty instead of the residual scatter of an artificially smooth curve."""
    if not lane_data or len(lane_data['f']) < 2:
        return None
    t = pd.to_datetime(lane_data['t'])
    f = np.asarray(lane_data['f'], float)
    ts = (t - t0).total_seconds().to_numpy()
    o = np.argsort(ts)
    ts, f = ts[o], f[o]
    tmin, tmax = ts.min(), ts.max()
    k = _thin(ts)
    if len(k) >= deg + 2:
        ts, f = ts[k], f[k]
    y = np.log10(f) if FIT_IN_LOGF else f
    sy = None
    if sigma_f:
        # d(log10 f) = df / (f ln10)
        sy = (sigma_f / (f * np.log(10))) if FIT_IN_LOGF else np.full_like(f, float(sigma_f))
    d = min(deg, len(ts) - 1)
    cov = None
    try:
        if sy is not None and np.all(np.isfinite(sy)) and np.all(sy > 0) and len(ts) > d + 1:
            p, cov = np.polyfit(ts, y, d, w=1.0 / sy, cov='unscaled')
        elif len(ts) >= d + 2:
            p, cov = np.polyfit(ts, y, d, cov=True)
        else:
            p = np.polyfit(ts, y, d)
    except (ValueError, np.linalg.LinAlgError):
        p = np.polyfit(ts, y, d)
    return {'p': p, 'cov': cov, 'tmin': tmin, 'tmax': tmax, 'n_fit': len(ts),
            'sigma_f': sigma_f}


def _coeffs(fit, rng=None, sample=False):
    if fit is None:
        return None
    if sample and rng is not None and fit['cov'] is not None:
        try:
            return rng.multivariate_normal(fit['p'], fit['cov'])
        except np.linalg.LinAlgError:
            return fit['p']
    return fit['p']


def _eval(fit, coeffs, tg):
    """Frequency [MHz] of a lane fit on the grid, blanking anything outside the traced span."""
    if fit is None or coeffs is None:
        return np.full_like(tg, np.nan)
    y = np.polyval(coeffs, tg)
    f = 10 ** y if FIT_IN_LOGF else y
    f[(tg < fit['tmin']) | (tg > fit['tmax'])] = np.nan
    return f


def lane_deriv(fit, ts):
    """Frequency, drift rate [MHz/s] and relative drift (1/f)(df/dt) [1/s] of a lane fit at ts.
    With a log fit, d(log10 f)/dt converts as df/dt = f ln(10) d(log10 f)/dt, and the relative
    drift is just ln(10) d(log10 f)/dt - a property of the shock, free of the band's harmonic."""
    dy = np.polyval(np.polyder(fit['p']), ts)
    if FIT_IN_LOGF:
        f = 10 ** np.polyval(fit['p'], ts)
        return f, f * np.log(10) * dy, np.log(10) * dy
    f = np.polyval(fit['p'], ts)
    return f, dy, dy / f


def pass_fits(pas):
    return {lab: _lane_fit(pas.get(lab), sigma_f=LANE_SIGMA.get(lab)) for lab in TRACED}


def order_lanes(labs):
    """Order a band's lanes by frequency over the interval where they OVERLAP in time.

    Comparing the mean frequency over each lane's own span is wrong as soon as the lanes cover
    different stretches of a drifting burst: a long lane that starts high and ends low can have
    a higher own-span mean than a short lane sitting above it, which inverts the split. The
    inversion is silent but fatal - it sends X = (f_U/f_L)^2 below 1, and M_A, v_A and B then
    all come out NaN."""
    fits = {l: _lane_fit(passes[0][l], sigma_f=LANE_SIGMA.get(l)) for l in labs}
    lo = max(f['tmin'] for f in fits.values())
    hi = min(f['tmax'] for f in fits.values())
    if hi > lo:
        ts = np.linspace(lo, hi, 40)
        note = f'compared over their {hi - lo:.0f} s of overlap'
    else:
        ts = np.linspace(min(f['tmin'] for f in fits.values()),
                         max(f['tmax'] for f in fits.values()), 40)
        note = ('WARNING: these lanes never overlap in time, so they are ordered on EXTRAPOLATED '
                'fits. Re-trace so the split branches share some time.')
    key = {l: float(np.nanmean(lane_deriv(fits[l], ts)[0])) for l in labs}
    return sorted(labs, key=lambda l: key[l]), key, note


def build_grid(passes, n=N_GRID):
    """One time grid spanning the whole burst - both bands belong to the same shock."""
    secs = []
    for pas in passes:
        for lab in TRACED:
            if pas.get(lab) and pas[lab]['f']:
                secs += list((pd.to_datetime(pas[lab]['t']) - t0).total_seconds())
    if not secs:
        raise RuntimeError('no traced points to build a time grid from')
    return np.linspace(np.nanmin(secs), np.nanmax(secs), n)


def _invert_grid(model, r_bounds=R_BOUNDS, nres=6000):
    rr = np.linspace(r_bounds[0], r_bounds[1], nres)
    return rr, model(rr)


def sg_smooth(y, window=9, poly=2):
    """Savitzky-Golay trend through a track. Interior gaps are bridged, but nothing is
    extrapolated past the traced span: np.interp clamps to the end values, which drew a flat
    horizontal trend across the part of the grid where the lane does not exist."""
    y = np.asarray(y, float)
    m = np.isfinite(y)
    if m.sum() < 5:
        return y
    idx = np.flatnonzero(m)
    inside = np.arange(idx[0], idx[-1] + 1)
    yy = np.interp(inside, idx, y[m])
    w = min(window, len(inside))
    if w % 2 == 0:
        w -= 1
    out = np.full_like(y, np.nan)
    out[inside] = yy if w < poly + 2 else savgol_filter(yy, w, poly)
    return out


def kin_degree(tg, span, deg=None):
    """Degree of the r(t) fit a lane can support. A quadratic over a short traced baseline is
    degenerate: its curvature is set by noise, and two branches of one band split can then come
    out with opposite accelerations. Below KIN_MIN_BASELINE_S the lane gets a straight line."""
    deg = KIN_DEG if deg is None else deg
    span = np.asarray(span, bool)
    if not span.any():
        return deg
    base = float(np.nanmax(tg[span]) - np.nanmin(tg[span]))
    return deg if base >= KIN_MIN_BASELINE_S else min(deg, 1)


def kinematics(r, tg, deg=None, span=None):
    """Speed [km/s] and acceleration [m/s^2] from a height track.

    The gate asks whether the density model resolves the lane, so it is judged against the
    lane's OWN traced span (`span`), not against the shared grid. Judging it against the grid
    silently threw away every short lane - a lane covering 4 of the burst's 32 minutes is a
    perfectly good measurement, it just covers less time."""
    v = np.full_like(tg, np.nan)
    a = np.full_like(tg, np.nan)
    good = np.isfinite(r)
    span = good if span is None else np.asarray(span, bool)
    deg = kin_degree(tg, span, deg)
    n_span = max(int(span.sum()), 1)
    r_range = (np.nanmax(r) - np.nanmin(r)) if good.sum() > 3 else 0.0
    if (good.sum() >= KIN_MIN_PTS and good.sum() / n_span >= KIN_MIN_FRAC
            and r_range > 0.01 and good.sum() > deg + 1):
        pr = np.polyfit(tg[good], r[good], min(deg, good.sum() - 1))
        v = np.polyval(np.polyder(pr, 1), tg) * R_SUN_M / 1e3
        a = np.polyval(np.polyder(pr, 2), tg) * R_SUN_M
        # blank the derivatives outside the span this track was traced over: each band covers a
        # different part of the burst, and a cubic extrapolated past its data runs away fast
        v[~good] = np.nan
        a[~good] = np.nan
    return np.where((v > 0) & (v < 3000), v, np.nan), a

In [ ]:
def realise_lanes(fits, tg, invert, rng=None, sample=False, mu=MU):
    """One realisation of height, speed, acceleration, Alfven speed and B for every lane, plus the
    combined F+H track. Each lane uses its band's harmonic number; B additionally needs that band's
    density jump X = (f_U/f_L)^2, so it only exists where the band has both split branches."""
    rr, ne = invert
    ne_min, ne_max = np.nanmin(ne), np.nanmax(ne)
    fval = {lab: _eval(fits.get(lab), _coeffs(fits.get(lab), rng, sample), tg) for lab in TRACED}
    out = {}
    for b in BANDS_TRACED:
        lo_lab, up_lab = SPLIT_PAIR[b]
        if up_lab is not None:
            with np.errstate(all='ignore'):
                X = (fval[up_lab] / fval[lo_lab]) ** 2
        else:
            X = np.full_like(tg, np.nan)
        MA = alfven_mach_from_X(X)
        for lab in LANE_ORDER[b]:
            ne_t = freq_to_density(fval[lab] * 1e6, harmonic=HARM[b])
            r = np.interp(ne_t, ne[::-1], rr[::-1])
            r[(ne_t > ne_max) | (ne_t < ne_min)] = np.nan
            v, a = kinematics(r, tg, span=np.isfinite(fval[lab]))
            vA = v / MA
            B = (vA * 1e3) * np.sqrt(MU0 * mu * M_P * (ne_t * 1e6)) * 1e4      # Gauss
            out[lab] = dict(r=r, v=v, a=a, vA=vA, B=B, X=X, MA=MA, ne=ne_t, f=fval[lab])

    # --- combined fundamental + harmonic track -------------------------------------------
    # the two bands are the same shock, so the upstream branches' heights and densities are
    # averaged where both exist and taken singly elsewhere: one trajectory across the whole burst
    ups = [out[SPLIT_PAIR[b][0]] for b in BANDS_TRACED if SPLIT_PAIR[b][0] in out]
    if MAKE_JOINT and len(ups) > 1:
        stack = lambda key: np.nanmean(np.vstack([u[key] for u in ups]), axis=0)
        r_j, ne_j, MA_j = stack('r'), stack('ne'), stack('MA')
        v_j, a_j = kinematics(r_j, tg)
        vA_j = v_j / MA_j
        B_j = (vA_j * 1e3) * np.sqrt(MU0 * mu * M_P * (ne_j * 1e6)) * 1e4
        out[JOINT] = dict(r=r_j, v=v_j, a=a_j, vA=vA_j, B=B_j, X=stack('X'), MA=MA_j,
                          ne=ne_j, f=np.full_like(tg, np.nan))
    return out


def grid_scalar(d, key):
    """Grid-average of a track's quantity and its combined standard error."""
    m, se = d[key + '_mean'], d[key + '_se']
    good = np.isfinite(m)
    if good.sum() == 0:
        return np.nan, np.nan
    return np.nanmean(m[good]), np.sqrt(np.nanmean(se[good] ** 2))


AGG_KEYS = ('r', 'v', 'a', 'vA', 'B', 'X', 'MA', 'ne', 'f')

def aggregate_lanes(passes, tg, model, n_mc=N_MC, seed=0):
    """Monte-Carlo aggregation per track, combining the fitting and repeat errors."""
    rng = np.random.default_rng(seed)
    invert = _invert_grid(model)
    fitsets = [pass_fits(pas) for pas in passes]
    stacks = {k: {kk: [] for kk in AGG_KEYS} for k in ALL_TRACKS}
    for fits in fitsets:
        for _ in range(n_mc):
            res = realise_lanes(fits, tg, invert, rng=rng, sample=True)
            for key, d in res.items():
                for kk in AGG_KEYS:
                    stacks[key][kk].append(d[kk])
    out = {}
    npass = max(len(passes), 1)
    for key in ALL_TRACKS:
        if not stacks[key]['r']:
            continue
        out[key] = {}
        for kk in AGG_KEYS:
            M = np.vstack(stacks[key][kk])
            out[key][kk + '_mean'] = np.nanmean(M, axis=0)
            out[key][kk + '_sd'] = np.nanstd(M, axis=0)
            out[key][kk + '_se'] = out[key][kk + '_sd'] / np.sqrt(npass)
    return out


def scalar_summary(passes):
    """Model-independent scalars per band: drift rate, relative drift, density jump, Alfven Mach
    number and relative bandwidth, each as mean +/- standard error across the passes. The absolute
    drift of the harmonic is twice the fundamental's, but the RELATIVE drift (1/f)(df/dt) is the
    same quantity for both bands and is the one to compare."""
    def mse(arr):
        arr = np.asarray(arr, float)
        if arr.size == 0:
            return (np.nan, np.nan)
        se = np.nanstd(arr, ddof=1) / np.sqrt(len(arr)) if len(arr) > 1 else 0.0
        return np.nanmean(arr), se

    rows = {}
    for b in BANDS_TRACED:
        lo_lab, up_lab = SPLIT_PAIR[b]
        Xv, MAv, drift, relbw, reldrift = [], [], [], [], []
        for pas in passes:
            fl = _lane_fit(pas.get(lo_lab))
            fu = _lane_fit(pas.get(up_lab)) if up_lab else None
            if fl and fu:
                lo, hi = max(fl['tmin'], fu['tmin']), min(fl['tmax'], fu['tmax'])
                if hi > lo:
                    gg = np.linspace(lo, hi, 20)
                    fLv, fUv = lane_deriv(fl, gg)[0], lane_deriv(fu, gg)[0]
                    Xg = (fUv / fLv) ** 2
                    Xv.append(np.nanmean(Xg))
                    MAv.append(np.nanmean(alfven_mach_from_X(Xg)))
                    relbw.append(np.nanmean((fUv - fLv) / fLv))
            if fl:
                gg = np.linspace(fl['tmin'], fl['tmax'], 20)
                _, dv, rd = lane_deriv(fl, gg)
                drift.append(np.nanmean(dv))                                  # MHz/s
                reldrift.append(np.nanmean(rd))                               # 1/s
        rows[b] = {'X': mse(Xv), 'M_A': mse(MAv), 'drift_MHz_s': mse(drift),
                   'rel_drift_s': mse(reldrift), 'rel_bandwidth': mse(relbw)}
    return rows

In [ ]:
# --- which lane of each band is the upstream branch, decided over their overlap -------------
LANE_ORDER, LANE_FREQ, LANE_ROLE = {}, {}, {}
for b in BANDS_TRACED:
    labs = [l for l in TRACED if LANE_BAND[l] == b]
    LANE_ORDER[b], key, note = order_lanes(labs)
    LANE_FREQ.update(key)
    for i, lab in enumerate(LANE_ORDER[b]):
        LANE_ROLE[lab] = ('upstream (lower)' if i == 0 else
                          'downstream (upper)' if i == 1 else 'extra')
    print(f'{BAND_NAME[b]} (s={HARM[b]}), {note}:')
    for lab in LANE_ORDER[b]:
        print(f'    {lab:12s} {LANE_FREQ[lab]:6.1f} MHz  ->  {LANE_ROLE[lab]}')
    if len(labs) < 2:
        print('    only one lane on this band, so it has no band split, hence no X or M_A')
SPLIT_PAIR = {b: (v[0], v[1] if len(v) > 1 else None) for b, v in LANE_ORDER.items()}
ALL_TRACKS = list(TRACED) + ([JOINT] if MAKE_JOINT else [])
print(f'\nanalysing {len(TRACED)} lane(s) separately'
      + ('; plus the combined F+H track (MAKE_JOINT = True)' if MAKE_JOINT
         else '. Set MAKE_JOINT = True to add the combined F+H track as well.'))

tg = build_grid(passes)
ALref = aggregate_lanes(passes, tg, MODEL_GRID[REF_MODEL_NAME])
scalars = scalar_summary(passes)
TRACKS = [k for k in ALL_TRACKS if k in ALref]

print(f'\nReference density model: {REF_MODEL_NAME}\n')
print('model-independent scalars, measured separately from each band')
for b in BANDS_TRACED:
    s = scalars[b]
    print(f'  {BAND_NAME[b]} (s={HARM[b]}):  '
          f'drift = {s["drift_MHz_s"][0]:+.4f} +/- {s["drift_MHz_s"][1]:.4f} MHz/s,'
          f'  rel. drift = {s["rel_drift_s"][0]:+.5f} +/- {s["rel_drift_s"][1]:.5f} 1/s,'
          f'  X = {s["X"][0]:.3f} +/- {s["X"][1]:.3f},'
          f'  M_A = {s["M_A"][0]:.3f} +/- {s["M_A"][1]:.3f},'
          f'  BDW = {s["rel_bandwidth"][0]:.3f} +/- {s["rel_bandwidth"][1]:.3f}')

for b in BANDS_TRACED:
    x = scalars[b]['X'][0]
    if np.isfinite(x) and x < 1:
        print(f'  ERROR: X < 1 for the {BAND_NAME[b]} band. The split branches are the wrong way '
              'round, so M_A, v_A and B will all be NaN. Check the ordering printed above '
              'against the traced-lanes figure.')

KIN_DEG_USED, KIN_BASELINE = {}, {}
for lab in TRACED:
    ff = _fits_A4.get(lab) if '_fits_A4' in dir() else None
    d = ALref.get(lab)
    if d is None:
        continue
    sp = np.isfinite(d['f_mean'])
    KIN_BASELINE[lab] = float(np.nanmax(tg[sp]) - np.nanmin(tg[sp])) if sp.any() else 0.0
    KIN_DEG_USED[lab] = kin_degree(tg, sp)

print(f'\nmodel-dependent quantities on {REF_MODEL_NAME}')
for lab, dg in KIN_DEG_USED.items():
    if dg < KIN_DEG:
        print(f'  NOTE: {lab} spans only {KIN_BASELINE[lab]:.0f} s (< KIN_MIN_BASELINE_S = '
              f'{KIN_MIN_BASELINE_S} s), so r(t) is fitted with a STRAIGHT LINE. Its speed is '
              'measured; its acceleration is not and is reported as zero.')
print('  (each lane is averaged over its OWN traced interval, so a decelerating shock gives a')
print('   higher mean speed for a lane that covers the earlier part of the burst)')
for key in TRACKS:
    d = ALref[key]
    if np.isfinite(d['r_mean']).sum() <= 2:
        print(f'  {key:12s}:  no height solution for this model')
        continue
    print(f'  {key:12s}:  '
          f'r = {np.nanmean(d["r_mean"]):.3f} +/- {np.nanmean(d["r_se"]):.3f} Rsun,'
          f'  v_sh = {np.nanmean(d["v_mean"]):.0f} +/- {np.nanmean(d["v_se"]):.0f} km/s,'
          f'  v_A = {np.nanmean(d["vA_mean"]):.0f} km/s,'
          f'  B = {np.nanmean(d["B_mean"]):.3f} G')

> **Output 1 &mdash; Lane roles and the model-independent scalars.** (printed above)
>
> First block: which lane of each band is the **upstream** (lower-frequency) branch of its split and
> which is the **downstream** branch, together with the frequencies they were compared at and the
> length of the interval over which the comparison was made.
>
> Second block: the drift rate, the relative drift $f^{-1}\dot f$, the density jump
> $X=(f_U/f_L)^2$, the Alfv&eacute;n Mach number $M_A$ and the relative bandwidth &mdash; per band,
> and **free of any density model**.
>
> Third block: height, shock speed, Alfv&eacute;n speed and magnetic field per lane on the
> reference model, with a note naming any lane whose traced baseline is too short for a quadratic
> $r(t)$ and which was therefore fitted with a straight line.

## A.5 &middot; Checking the fundamental / harmonic identification

The two bands are assumed to be $s=1$ and $s=2$ of one shock. Four checks on that, all made
against the data rather than assumed.

**1. Frequency ratio over the overlap.** Where both bands are traced, the ratio of their upstream
branches must be 2. Comparing the harmonic's upstream branch against the fundamental's *downstream*
branch instead gives $2/(1+\mathrm{BDW})$, so the branches have to be paired correctly for this
test to mean anything &mdash; which is why &sect;A.4 sorts them in frequency.

**2. Relative drift rate.** The absolute drift $\mathrm{d}f/\mathrm{d}t$ of the harmonic is twice
the fundamental's, purely because its frequency is twice as high. The **relative** drift
$f^{-1}\mathrm{d}f/\mathrm{d}t$ removes that factor and is a property of the shock alone, so the two
bands must return the same value &mdash; evaluated over the same interval, since it changes through
the burst.

**3. Height agreement.** Converting each band with its own $s$ must place both on the same
height-time trajectory. This is not an independent test of the density model &mdash; any model maps
$f_H/2$ and $f_F$ to the same density &mdash; but it does verify the harmonic assignment and shows
how well the two segments join where they overlap.

**4. Circular polarisation.** The Stokes V/I frame is sampled in a
$\pm$`POL_DT_S`&nbsp;s&nbsp;&times;&nbsp;$\pm$`POL_DF_MHZ`&nbsp;MHz box along every traced lane.
The number to compare between bands is the **signed** mean. Averaging $|V/I|$ instead measures the
noise, not the polarisation: for a signal at or below the noise, $\langle|x|\rangle\to0.8\sigma$
whatever the true mean is, so two bands with opposite signed polarisation return the same
$|V/I|$. Both are tabulated, but only the signed mean is used for the comparison.
Fundamental plasma emission escapes in the o-mode and is expected to be the more strongly polarised
of the two; harmonic emission comes from a coalescence that largely cancels the sense of
polarisation. Type II polarisation is weak in absolute terms and its sense depends on the
line-of-sight field, so this supports the identification rather than settling it &mdash; but a
systematic contrast between the two bands in the same event, on the same instrument, is meaningful.

In [ ]:
POL_T = POL.index.to_numpy()
POL_F = np.asarray(POL.columns, float)
POL_V = POL.to_numpy()


def sample_polarisation(t_list, f_list, dt_s=POL_DT_S, df_mhz=POL_DF_MHZ):
    """Median Stokes V/I in a (+/- dt_s, +/- df_mhz) box around each traced (t, f) point, taken
    from the decimated V/I layer."""
    tv = pd.to_datetime(pd.Series(t_list)).to_numpy()
    fv = np.asarray(f_list, float)
    half = np.timedelta64(int(dt_s * 1e3), 'ms')
    out = np.full(len(fv), np.nan)
    for k, (tt, ff) in enumerate(zip(tv, fv)):
        i0 = np.searchsorted(POL_T, tt - half)
        i1 = max(np.searchsorted(POL_T, tt + half), i0 + 1)
        j = np.abs(POL_F - ff) <= df_mhz
        if j.any():
            blk = POL_V[i0:i1][:, j]
            if blk.size:
                out[k] = np.nanmedian(blk)
    return out


pol_rows = []
POL_TRACK = {}
for lab in TRACED:
    ref = next(p[lab] for p in passes if p.get(lab, {}).get('f'))
    # resample the lane fit on a dense grid so the polarisation is not read at the click points only
    fit = _lane_fit(ref)
    ts = np.linspace(fit['tmin'], fit['tmax'], 120)
    tt = [t0 + pd.Timedelta(seconds=float(s)) for s in ts]
    ff = lane_deriv(fit, ts)[0]
    p = sample_polarisation(tt, ff)
    POL_TRACK[lab] = dict(t=pd.to_datetime(tt), f=ff, p=p)
    n = int(np.isfinite(p).sum())
    pol_rows.append({'lane': lab, 'band': LANE_BAND[lab], 'role': LANE_ROLE[lab],
                     'V_over_I_mean': np.nanmean(p), 'V_over_I_sd': np.nanstd(p),
                     'V_over_I_sem': np.nanstd(p) / np.sqrt(max(n, 1)),
                     'abs_V_over_I_mean': np.nanmean(np.abs(p)),
                     'abs_V_over_I_max': np.nanmax(np.abs(p)), 'n_samples': n})

pol_table = pd.DataFrame(pol_rows)
pol_table.to_csv(os.path.join(OUTDIR, 'typeii_polarisation.csv'), index=False)

# The SIGNED mean is the physical quantity. Taking |V/I| first and then averaging measures the
# noise, not the polarisation: for a signal buried in noise, mean|x| -> 0.8 sigma whatever the
# true mean is, so two bands with opposite signed polarisation report the same |V/I|.
pF = pol_table[pol_table['band'] == 'F']['V_over_I_mean'].mean()
pH = pol_table[pol_table['band'] == 'H']['V_over_I_mean'].mean()
noise = pol_table['V_over_I_sd'].median()
print(f'signed mean V/I:  fundamental {pF:+.4f},  harmonic {pH:+.4f}   '
      f'(point-to-point scatter along a lane {noise:.4f})')
if abs(pF - pH) < 0.5 * noise:
    print('  the two bands differ by less than half the scatter, so the contrast is not '
          'significant here')
elif abs(pF) > abs(pH):
    print('  the fundamental is the more strongly polarised, as expected for o-mode '
          'fundamental emission')
else:
    print('  the harmonic is the more strongly polarised - unusual, worth a second look at the '
          'tracing and at the RFI stripes in panel (a)')
print(f'mean |V/I| is also tabulated but do NOT use it to compare the bands: for |V/I| ~ the '
      f'noise it just returns the noise level ({0.8 * noise:.4f} here).')
pol_table.round(4)

> **Table 3 &mdash; Circular polarisation along each lane.** `typeii_polarisation.csv`
>
> The Stokes V/I frame sampled in a $\pm$`POL_DT_S`&nbsp;s&nbsp;$\times$&nbsp;$\pm$`POL_DF_MHZ`&nbsp;MHz
> box centred on every point of the fitted lane: the signed mean, its scatter and standard error,
> the mean and maximum of $|V/I|$, and the number of samples.
>
> *Use the signed mean, not $|V/I|$.* For a signal at or below the noise
> $\langle|x|\rangle\rightarrow0.8\sigma$ regardless of the true mean, so $|V/I|$ returns the noise
> level for every band and destroys exactly the contrast you are looking for. Fundamental emission
> escapes in the o-mode and is expected to be the more strongly polarised of the two.

In [ ]:
# ---- tests 1-3: ratio, relative drift and height agreement over the overlap ----
_fits = pass_fits(passes[0])
_rr, _ne = _invert_grid(MODEL_GRID[REF_MODEL_NAME])
fh_rows = []


def _height(fit, band, ts):
    f = lane_deriv(fit, ts)[0]
    ne_t = freq_to_density(f * 1e6, harmonic=HARM[band])
    r = np.interp(ne_t, _ne[::-1], _rr[::-1])
    r[(ne_t > np.nanmax(_ne)) | (ne_t < np.nanmin(_ne))] = np.nan
    return f, r


fF = _fits.get(SPLIT_PAIR['F'][0]) if 'F' in SPLIT_PAIR else None
fH = _fits.get(SPLIT_PAIR['H'][0]) if 'H' in SPLIT_PAIR else None
OVERLAP = None
if fF is None or fH is None:
    print('both bands need at least one traced lane for the F/H consistency tests - skipped')
else:
    lo, hi = max(fF['tmin'], fH['tmin']), min(fF['tmax'], fH['tmax'])
    OVERLAP = (lo, hi) if hi > lo else None
    if OVERLAP is None:
        print(f'the two bands do not overlap in time ({lo - hi:.0f} s apart); the ratio below is '
              'EXTRAPOLATED and is much weaker evidence. Re-trace so the bands share some time '
              'if the spectrum allows it.')
        lo, hi = min(fF['tmin'], fH['tmin']), max(fF['tmax'], fH['tmax'])
    ts = np.linspace(lo, hi, 60)
    f_F, r_F = _height(fF, 'F', ts)
    f_H, r_H = _height(fH, 'H', ts)
    ratio = f_H / f_F
    dres = r_H - r_F
    # the relative drift must be compared ON THE OVERLAP: it changes through the burst, and the
    # band-averaged values in `scalars` are averages over two different stretches of it
    rdF = np.nanmean(lane_deriv(fF, ts)[2])
    rdH = np.nanmean(lane_deriv(fH, ts)[2])

    print(f'{SPLIT_PAIR["F"][0]} vs {SPLIT_PAIR["H"][0]}, '
          f'{(t0 + pd.Timedelta(seconds=lo)).strftime("%H:%M:%S")}'
          f'-{(t0 + pd.Timedelta(seconds=hi)).strftime("%H:%M:%S")} UT  ({hi - lo:.0f} s)\n')
    print(f'1. frequency ratio   f_H / f_F = {np.nanmean(ratio):.3f} +/- {np.nanstd(ratio):.3f}'
          '        (2.000 expected)')
    print(f'2. relative drift    over the overlap, F: {rdF:+.5f} 1/s,   H: {rdH:+.5f} 1/s   '
          f'(differ by {100 * abs(rdH - rdF) / abs(rdF):.1f}%, must agree)')
    print(f'3. height residual   r_H - r_F = {np.nanmean(dres):+.4f} +/- {np.nanstd(dres):.4f} Rsun'
          f'   ({100 * np.nanmean(np.abs(dres)) / np.nanmean(r_F):.2f}% of the height)')
    print(f'4. band splits       X_F = {scalars["F"]["X"][0]:.3f} +/- {scalars["F"]["X"][1]:.3f},   '
          f'X_H = {scalars["H"]["X"][0]:.3f} +/- {scalars["H"]["X"][1]:.3f}   '
          '(same shock -> same density jump)')

    # Each row is self-contained. `kind` says whether it is a raw MEASUREMENT or a
    # CONSISTENCY TEST; only tests have an `expected` value and an n_sigma, and a blank expected
    # means the row is a measurement with nothing to compare against, not a missing number.
    def _row(kind, test, value, error, expected=np.nan, note=''):
        n = (abs(value - expected) / error) if (np.isfinite(expected) and np.isfinite(error)
                                                and error > 0) else np.nan
        return {'kind': kind, 'test': test, 'value': value, 'error': error,
                'expected': expected, 'n_sigma': n,
                'verdict': ('' if not np.isfinite(n) else
                            ('FLAG' if n > CONSIST_SIGMA else 'ok')), 'note': note}

    _eX = np.hypot(scalars['F']['X'][1], scalars['H']['X'][1])
    fh_rows = [
        _row('consistency test', 'f_H / f_F over the overlap', float(np.nanmean(ratio)),
             float(np.nanstd(ratio)), 2.0,
             'must be 2 if the two bands are the fundamental and its harmonic'),
        _row('consistency test', 'relative drift, H minus F [1/s]', rdH - rdF,
             0.02 * abs(rdF), 0.0,
             'f^-1 df/dt is a property of the shock, so both bands must give the same value; '
             'the error is a nominal 2% of the F value, there is no independent estimate'),
        _row('consistency test', 'height residual r_H - r_F [Rsun]', float(np.nanmean(dres)),
             float(np.nanstd(dres)), 0.0,
             'converting each band with its own harmonic number must put them at the same height'),
        _row('consistency test', 'density jump, X_H minus X_F',
             scalars['H']['X'][0] - scalars['F']['X'][0], _eX, 0.0,
             'X is model-independent and both bands measure the same shock'),
        _row('measurement', 'relative drift F, overlap [1/s]', rdF, np.nan, np.nan,
             'evaluated from one fit at the overlap; no independent error'),
        _row('measurement', 'relative drift H, overlap [1/s]', rdH, np.nan, np.nan,
             'evaluated from one fit at the overlap; no independent error'),
        _row('measurement', 'density jump X (F)', *scalars['F']['X'], np.nan,
             'from the F band split alone'),
        _row('measurement', 'density jump X (H)', *scalars['H']['X'], np.nan,
             'from the H band split alone'),
        _row('measurement', 'signed mean V/I (F)', pF, np.nan, np.nan,
             'scatter along the lane is in typeii_polarisation.csv'),
        _row('measurement', 'signed mean V/I (H)', pH, np.nan, np.nan,
             'scatter along the lane is in typeii_polarisation.csv'),
    ]

fh_test = pd.DataFrame(fh_rows)
if len(fh_test):
    fh_test.to_csv(os.path.join(OUTDIR, 'fundamental_harmonic_tests.csv'), index=False)
    print('\nkind = "consistency test" rows have an expected value and an n_sigma; '
          '"measurement" rows do not.')
    print('a blank expected/n_sigma means there is nothing to compare that number against, '
          'not a missing result.')
fh_test.round(4)

> **Table 4 &mdash; Fundamental / harmonic consistency.** `fundamental_harmonic_tests.csv`
>
> `kind` separates the two sorts of row. A **consistency test** compares two measurements of the
> same physical quantity and therefore has an `expected` value, an `n_sigma` and a verdict:
> $f_H/f_F$ must be 2, the two bands' relative drifts must be equal, the height residual
> $r_H-r_F$ must be zero, and $X_H-X_F$ must be zero. A **measurement** row is a raw number with
> nothing to compare it against, so its `expected` and `n_sigma` are deliberately blank &mdash;
> that is not a missing result.
>
> *This is the check that the whole F/H interpretation rests on.* A `FLAG` here means a lane is
> mis-traced or a branch is misassigned, and it must be resolved before any of these numbers are
> used.

In [ ]:
# ---- consistency audit: quantities that MUST agree if the physics is right ----------------
# Everything below compares two measurements of the same thing. A disagreement beyond
# CONSIST_SIGMA is not a rounding issue, it means a lane is mis-traced or a branch is misassigned,
# and it has to be resolved before any of these numbers go into a paper.
def _pair(a, ea, b, eb):
    d = a - b
    ed = np.hypot(ea if np.isfinite(ea) else 0.0, eb if np.isfinite(eb) else 0.0)
    return d, ed, (abs(d) / ed if ed > 0 else np.inf)


audit = []


def _check(name, a, ea, b, eb, what, tol=CONSIST_SIGMA):
    d, ed, n = _pair(a, ea, b, eb)
    audit.append({'check': name, 'value_1': a, 'value_2': b, 'difference': d,
                  'combined_error': ed, 'n_sigma': n, 'verdict': 'FLAG' if n > tol else 'ok',
                  'why_it_matters': what})


# 1. the two bands are the same shock, so the density jump and the relative drift must match
if len(BANDS_TRACED) > 1 and all(np.isfinite(scalars[b]['X'][0]) for b in BANDS_TRACED):
    _check('density jump X: F vs H', *scalars['F']['X'], *scalars['H']['X'],
           'X is model-independent and both bands measure the same shock')
if OVERLAP is not None:
    _check('relative drift over the overlap: F vs H', rdF, abs(rdF) * 0.02, rdH,
           abs(rdH) * 0.02, 'f^-1 df/dt is a property of the shock, free of the harmonic number')

# 2. within a band the two split branches straddle one shock front, so their kinematics and B
#    must be close; a large disagreement means one branch is not where you think it is
for b in BANDS_TRACED:
    lo, up = SPLIT_PAIR[b]
    if up is None or lo not in ALref or up not in ALref:
        continue
    for key, lab in [('v', 'shock speed'), ('a', 'acceleration'), ('B', 'magnetic field')]:
        if key == 'a' and min(KIN_DEG_USED.get(lo, KIN_DEG),
                              KIN_DEG_USED.get(up, KIN_DEG)) < KIN_DEG:
            continue          # one of them has no measured acceleration, so there is nothing
                              # to compare - a straight-line fit reports a = 0 by construction
        m1, e1 = grid_scalar(ALref[lo], key)
        m2, e2 = grid_scalar(ALref[up], key)
        if np.isfinite(m1) and np.isfinite(m2):
            _check(f'{lab}: {lo} vs {up}', m1, e1, m2, e2,
                   'the two split branches of one band bracket the same shock front')

audit = pd.DataFrame(audit)
if len(audit):
    audit.to_csv(os.path.join(OUTDIR, 'consistency_audit.csv'), index=False)
    for _, r in audit[audit.verdict == 'FLAG'].iterrows():
        print(f'  *** FLAG: {r["check"]} differ by {r["n_sigma"]:.1f} sigma '
              f'({r["value_1"]:.3g} vs {r["value_2"]:.3g}). {r["why_it_matters"]}.')
    if not (audit.verdict == 'FLAG').any():
        print('  all consistency checks pass')

# 3. is the acceleration meaningful at all? It is the second derivative of a height track whose
#    curvature is set largely by the assumed density profile, so it carries a systematic far
#    larger than its statistical error, and over a short traced baseline it is unconstrained
print()
print('acceleration reality check (statistical significance is NOT the issue here):')
for lab in TRACED:
    d = ALref.get(lab)
    if d is None:
        continue
    a_, ea_ = grid_scalar(d, 'a')
    if not np.isfinite(a_):
        continue
    base = KIN_BASELINE.get(lab, np.nan)
    if KIN_DEG_USED.get(lab, KIN_DEG) < KIN_DEG:
        print(f'  {lab:10s} NOT MEASURED over {base:5.0f} s of baseline - r(t) was fitted with a '
              'straight line, so a = 0 by construction')
        continue
    tag = []
    if abs(a_) > A_PLAUSIBLE_MS2:
        tag.append(f'|a| = {abs(a_):.0f} m/s^2 is {abs(a_) / 50:.0f}x a typical coronal value')
    print(f'  {lab:10s} a = {a_:+8.1f} +/- {ea_:5.1f} m/s^2 over {base:5.0f} s'
          + ('   <-- ' + '; '.join(tag) if tag else '   (plausible)'))
print('a is dominated by the density model, not by the measurement: see the model x fold range '
      'in A.8 before quoting it.')

> **Table 5 &mdash; Consistency audit.** `consistency_audit.csv`
>
> Every pair of quantities that must agree if the physics is right, with the difference, the
> combined error, the significance and a verdict. Two families: **between bands** (the density jump
> and the relative drift, both model-independent), and **within a band** (the two split branches
> straddle one shock front, so their speed, acceleration and $B$ must be close).
>
> Below it, an **acceleration reality check**: $|a|$ against a plausibility threshold, the traced
> baseline it came from, and an explicit `NOT MEASURED` for any lane whose $r(t)$ was fitted with a
> straight line. Statistical significance is not the issue for $a$ &mdash; its curvature is set
> largely by the assumed density profile, so it carries a systematic far larger than its error bar.

In [ ]:
fig = plt.figure(figsize=[15, 11])

# (a) the V/I spectrogram with every traced lane on top
ax = fig.add_subplot(311)
step = max(1, POL.shape[0] // PREVIEW_MAX_TCOLS)
pv = np.nanpercentile(np.abs(POL.to_numpy()), 99)
pm = ax.pcolormesh(POL.index[::step], np.asarray(POL.columns, float), POL.to_numpy().T[:, ::step],
                   vmin=-pv, vmax=pv, cmap='seismic', rasterized=True)
fig.colorbar(pm, ax=ax, pad=0.01, label='Stokes V/I')
for lab in TRACED:
    ax.plot(POL_TRACK[lab]['t'], POL_TRACK[lab]['f'], color='k', lw=1.8)
    ax.plot(POL_TRACK[lab]['t'], POL_TRACK[lab]['f'], color=LANE_COL[lab], lw=1.1, label=lab)
if INVERT_FREQ:
    ax.invert_yaxis()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
ax.set_ylabel('Frequency (MHz)')
ax.set_title('(a) Circular polarisation with the traced lanes overlaid')
ax.legend(fontsize=8, ncol=4, loc='lower left')

# (b) V/I along each lane, F against H
ax = fig.add_subplot(312)
for lab in TRACED:
    d = POL_TRACK[lab]
    ax.plot(d['t'], d['p'], 'o', ms=3, color=LANE_COL[lab], alpha=0.45)
    ax.plot(d['t'], sg_smooth(d['p'], window=11), '-', lw=2, color=LANE_COL[lab],
            label=f'{lab}: {np.nanmean(d["p"]):+.3f} $\\pm$ {np.nanstd(d["p"]):.3f}')
ax.axhline(0, color='0.6', lw=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
ax.set_ylabel('Stokes V/I along the lane')
ax.set_title('(b) Degree of circular polarisation: signed mean $V/I$ = '
             f'{pF:+.3f} (F) vs {pH:+.3f} (H)')
ax.legend(fontsize=8, ncol=4)
ax.grid(alpha=0.3)

# (c) the two height tracks, which must be one trajectory
ax = fig.add_subplot(313)
for b, col in zip(BANDS_TRACED, ['navy', 'firebrick']):
    fit = _fits.get(SPLIT_PAIR[b][0])
    if fit is None:
        continue
    ts = np.linspace(fit['tmin'], fit['tmax'], 80)
    _, r = _height(fit, b, ts)
    ax.plot(ts, r, '-', lw=2.4, color=col, label=f'{BAND_NAME[b]} ($s$ = {HARM[b]})')
d = ALref.get(JOINT)
if d is not None:
    ax.errorbar(tg, d['r_mean'], yerr=d['r_se'], fmt='o', ms=3.5, color='0.35', capsize=2,
                alpha=0.75, label=JOINT, zorder=0)
if OVERLAP is not None:
    ax.axvspan(*OVERLAP, color='0.85', alpha=0.6, zorder=-1)
    ax.text(np.mean(OVERLAP), ax.get_ylim()[0], 'overlap', ha='center', va='bottom', fontsize=9)
ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
ax.set_ylabel(r'$r\,/\,R_\odot$')
ax.set_title(f'(c) Both bands on one height-time trajectory ({REF_MODEL_NAME})')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

fig.tight_layout()
save_fig(fig, 'fundamental_harmonic_checks')
plt.show()

> **Figure 5 &mdash; Fundamental / harmonic checks.** `fundamental_harmonic_checks.png`
>
> *(a)* The Stokes V/I dynamic spectrum with every traced lane overlaid, so you can see at a glance
> whether a lane crosses an RFI stripe. *(b)* The degree of circular polarisation along each lane,
> points and a Savitzky-Golay trend, with the signed mean and scatter per lane in the legend.
> *(c)* Both bands converted to height with their own harmonic number, on one axis, with the
> combined track and the interval in which the two overlap shaded.
>
> *What to look for:* in (c) the two segments must lie on a single continuous trajectory. They
> will, for any density model, provided the harmonic assignment is right &mdash; that is what makes
> this a test of the identification rather than of the model.

## A.6 &middot; Kinematics and coronal magnetic field

Height, shock speed, acceleration and $B$ for every traced lane and for the combined `F+H joint`
track, on the reference model, with the Monte-Carlo $\pm$SEM error bars and a Savitzky-Golay trend
through each. The legend quotes the grid-averaged value. The joint track is the one to quote: it
uses the fundamental early in the burst and the harmonic late, and so covers the full interval over
which the shock was observed.

Two things to read carefully here. **The error bars are statistical only** &mdash; the tracing
repeats plus the polynomial covariance. They are small because the height is a weak function of
frequency in an exponential corona; the uncertainty that actually matters is the density model,
and that is the model&times;fold range in &sect;A.8. Quote both. **Acceleration is constant per
track by construction**, because `KIN_DEG = 2` fits $r(t)$ with a quadratic. That is deliberate: a
cubic bends the height track into an S and then reports a spurious $\sim10^3$ m s$^{-2}$ near the
ends. If you want a time-dependent acceleration, take it from the Byrne fit in &sect;A.7, which is
non-parametric, or raise `KIN_DEG` knowing what it does.

Each track is drawn only over the interval it was traced; nothing is extrapolated.

In [ ]:
def _has(key, k):
    return key in ALref and np.isfinite(ALref[key][k + '_mean']).sum() > 2


def track_panel(ax, key_list, k, ylabel, unit, fmt='{:.3g}'):
    for key in key_list:
        if not _has(key, k):
            continue
        c = LANE_COL[key]
        m, err = ALref[key][k + '_mean'], ALref[key][k + '_se']
        joint = key == JOINT
        ax.errorbar(tg, m, yerr=err, fmt='o', ms=3, color=c, ecolor=c, elinewidth=0.7,
                    capsize=1.5, alpha=(0.6 if joint else 0.4))
        ax.plot(tg, sg_smooth(m), '-', color=c, lw=(2.6 if joint else 1.6),
                zorder=(5 if joint else 2),
                label=(f'{key}: {fmt.format(np.nanmean(m))} $\\pm$ '
                       f'{fmt.format(np.sqrt(np.nanmean(err ** 2)))} {unit}'))
    ax.set_ylabel(ylabel)
    ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)


fig = plt.figure(figsize=[14, 9])

ax = fig.add_subplot(221)
track_panel(ax, TRACKS, 'r', r'$r\,/\,R_\odot$', r'$R_\odot$', '{:.3f}')
ax.set_title(f'(a) Height-time  ({REF_MODEL_NAME})')

ax = fig.add_subplot(222)
track_panel(ax, TRACKS, 'v', r'$v_{\rm sh}$ [km s$^{-1}$]', 'km/s', '{:.0f}')
ax.set_title('(b) Shock speed')

ax = fig.add_subplot(223)
track_panel(ax, TRACKS, 'a', r'$a$ [m s$^{-2}$]', r'm s$^{-2}$', '{:.1f}')
ax.axhline(0, color='0.6', lw=0.8)
ax.set_title('(c) Acceleration')

ax = fig.add_subplot(224)
track_panel(ax, TRACKS, 'B', r'$B$ [G]', 'G', '{:.3f}')
ax.set_title('(d) Coronal magnetic field')
if not any(_has(k, 'B') for k in TRACKS):
    # B needs M_A, which needs 1 <= X < 4. Say so in the panel instead of leaving it blank.
    bad = [f'{BAND_NAME[b]}: X = {scalars[b]["X"][0]:.3f}' for b in BANDS_TRACED
           if not np.isfinite(scalars[b]['M_A'][0])]
    ax.text(0.5, 0.5, 'no B: it needs $M_A$, which is only defined for $1 \\leq X < 4$\n'
            + ('\n'.join(bad) if bad else 'no band split traced'),
            transform=ax.transAxes, ha='center', va='center', fontsize=10, color='firebrick')

fig.suptitle(f'Type II kinematics and coronal magnetic field, NenuFAR {EVENT_DATE}\n'
             'error bars are STATISTICAL only; the density-model systematic is in A.8',
             y=1.03, fontsize=12)
fig.tight_layout()
save_fig(fig, 'typeii_kinematics_Bfield')
plt.show()

> **Figure 6 &mdash; Kinematics and coronal magnetic field.** `typeii_kinematics_Bfield.png`
>
> Per lane, on the reference density model: *(a)* height, *(b)* shock speed, *(c)* acceleration,
> *(d)* coronal magnetic field. Points carry the Monte-Carlo error, the line is a Savitzky-Golay
> trend, and the legend quotes each lane's grid-averaged value. Each lane is drawn only over the
> interval it was actually traced; nothing is extrapolated.
>
> *Two things to keep in mind.* The error bars are **statistical only** &mdash; the density-model
> systematic is Figure&nbsp;8, and it is far larger. And the acceleration is **constant per lane by
> construction**, because $r(t)$ is fitted with a quadratic (`KIN_DEG = 2`); lanes shorter than
> `KIN_MIN_BASELINE_S` get a straight line and no acceleration at all.

## A.7 &middot; Height-time fitting: polynomial vs Gallagher vs Byrne

&sect;A.6 obtained $v$ and $a$ by differentiating a polynomial fit of $r(t)$. Because the
acceleration is the second derivative of a fitted curve, it is the quantity most exposed to the
choice of fitting function, so the **combined F+H height-time track** is refit three ways and
compared:

1. **Polynomial** &mdash; degree `POLY_DEG`, weighted by the height errors.
2. **Gallagher, Lawrence & Dennis (2003)** &mdash; a two-phase acceleration profile
$$a(t)=\left[\frac{1}{a_r e^{t/\tau_r}}+\frac{1}{a_d e^{-t/\tau_d}}\right]^{-1},$$
   with $v$ and $h$ obtained by integrating $a(t)$ numerically (the exact double integral, not an
   analytic approximation to it).
3. **Byrne et&nbsp;al. (2013)** &mdash; Savitzky-Golay smoothing of $h(t)$ with a bootstrap for the
   error band. This is the actual Byrne method; the arctangent height-time form sometimes
   attributed to that paper is a different thing and is not used here.

**A caveat on Gallagher that matters.** With $a_r,a_d>0$ &mdash; which the fit requires, or the
reciprocal sum diverges &mdash; the profile above is **strictly positive**. It was built to
describe the impulsive *acceleration* phase of an eruption, and it **cannot represent a
decelerating shock at all**. Fitted to a decelerating track it pins $a_r,a_d$ at their lower bound,
returns $a\simeq0$, and draws a straight line through curved data without complaint. The cell
below detects this &mdash; both by comparing $\chi^2_{\rm red}$ against the best method and by
checking whether Gallagher's $a(t)$ can go negative when the polynomial says it should &mdash; and
marks the method inapplicable. Most type II shocks decelerate once past the impulsive phase, so
**expect this warning to fire and do not quote Gallagher numbers when it does.**

Each fit is bootstrapped `N_BOOT` times by resampling the heights within their errors, giving the
shaded bands. Using the joint track matters here: the two bands separately each cover only part of
the burst, and a second derivative taken over a short segment is barely constrained.

**Nothing is finite differenced.** Each fit returns $h$, $v$ and $a$ as callables with analytic
derivatives: the polynomial is differentiated exactly, Gallagher's $a(t)$ *is* the model so $v$ and
$h$ are its exact integrals, and Byrne's derivatives come from the Savitzky-Golay filter itself
(`deriv=1`, `deriv=2`), which is the method as published. Differencing an interpolant instead
returns the slope of whichever segment the step lands in &mdash; a staircase in $v$ and delta
functions in $a$ &mdash; and differentiating a spline through SG-smoothed points twice makes the
acceleration ring.

**The bands: a parametric bootstrap.** Each fit is repeated `N_BOOT` times. On each repetition
every traced height is displaced by a draw from a Gaussian of its own 1$\sigma$ error,
$h_i\rightarrow h_i+\mathcal{N}(0,\sigma_i)$, and the curve is refitted from scratch. The band is
the standard deviation of those `N_BOOT` curves at each instant, so it propagates the height
uncertainty into $h$, $v$ and $a$ without assuming the fit is linear in its parameters. Note this
resamples from the *assumed noise model*, which is a parametric bootstrap &mdash; it is not the
non-parametric kind that resamples the data points with replacement, and it therefore inherits
whatever $\sigma_i$ you fed it. `band_scale` in the comparison table is the factor the band was multiplied by: it is
$\sqrt{\chi^2_{\rm red}}$ when `INFLATE_BY_CHI2` is on and the fit is worse than the errors
imply, and **1 otherwise, which is the default**. Inflating is the standard remedy for
underestimated errors, but for comparing models it hides the very thing you are looking for, so it
is off and `band_scale` should read 1 everywhere. A polynomial band is often narrower than its own line width: two free parameters
average the height scatter down hard, whereas Byrne is a local smoother and follows each
realisation, so its band is much the widest. That difference is information, not a plotting fault.

In [ ]:
RS_KM = R_SUN_M / 1e3

# which track the height-time fits and the model sweep use
if REF_LANE is not None:
    REF_TRACK = REF_LANE
elif MAKE_JOINT and JOINT in ALref and np.isfinite(ALref[JOINT]['r_mean']).sum() > 3:
    REF_TRACK = JOINT
else:
    _cand = [SPLIT_PAIR[b][0] for b in BANDS_TRACED] + list(TRACED)
    REF_TRACK = next((k for k in _cand
                      if k in ALref and np.isfinite(ALref[k]['r_mean']).sum() > 3), None)
if REF_TRACK is None:
    raise RuntimeError('no usable height-time track on ' + REF_MODEL_NAME)
print(f'height-time fits and the model sweep use: {REF_TRACK}  ({REF_MODEL_NAME})')


# Each fitter returns h, v and a as callables with ANALYTIC derivatives. Nothing is finite
# differenced: differencing an interpolant returns the slope of whichever segment the step lands
# in, which is where the staircase speeds and the 1e6 m/s^2 spikes came from.
def fit_polynomial(t, h, sig):
    """Weighted polynomial of degree POLY_DEG; v and a are its exact derivatives."""
    p = np.polyfit(t, h, POLY_DEG, w=(1.0 / sig if sig is not None else None))
    dp, ddp = np.polyder(p, 1), np.polyder(p, 2)
    return {'h': lambda tt: np.polyval(p, tt),
            'v': lambda tt: np.polyval(dp, tt),               # km/s
            'a': lambda tt: np.polyval(ddp, tt) * 1000}       # m/s^2


def gallagher_accel(t, ar, ad, tr, td):
    """Gallagher, Lawrence & Dennis (2003) reciprocal-sum acceleration profile [km/s^2]."""
    return 1.0 / (1.0 / (ar * np.exp(t / tr)) + 1.0 / (ad * np.exp(-t / td)))


def fit_gallagher(t, h, sig):
    """Gallagher et al. (2003): a(t) is the model, so v and h are its exact integrals and both
    derivatives come straight back out of the fitted parameters."""
    tgrid = np.linspace(t.min(), t.max(), 400)
    def model(tt, ar, ad, tr, td, h0, v0):
        acc = gallagher_accel(tgrid, ar, ad, tr, td)
        vv = v0 + cumulative_trapezoid(acc, tgrid, initial=0)
        hh = h0 + cumulative_trapezoid(vv, tgrid, initial=0)
        return np.interp(tt, tgrid, hh)
    v0g = (h[-1] - h[0]) / (t[-1] - t[0])
    p0 = [1e-3, 1e-3, 150, 150, h[0], v0g]
    lo = [1e-6, 1e-6, 20, 20, h[0] - 5e4, -2000]
    hi = [1e2, 1e2, 5e3, 5e3, h[0] + 5e4, 3000]
    popt, _ = curve_fit(model, t, h, p0=p0, sigma=sig, absolute_sigma=False,
                        bounds=(lo, hi), maxfev=200000)
    acc = gallagher_accel(tgrid, *popt[:4])
    vv = popt[5] + cumulative_trapezoid(acc, tgrid, initial=0)
    hh = popt[4] + cumulative_trapezoid(vv, tgrid, initial=0)
    return {'h': CubicSpline(tgrid, hh), 'v': CubicSpline(tgrid, vv),
            'a': lambda tt, _g=tgrid, _a=acc: np.interp(tt, _g, _a) * 1000}


def fit_byrne(t, h, sig):
    """Byrne et al. (2013): Savitzky-Golay. The SG filter returns the derivatives directly
    (deriv=1, 2), which is the method as published and is smooth. Differentiating a spline
    through the smoothed points instead makes the second derivative ring."""
    tu = np.linspace(t.min(), t.max(), max(len(t), 21))
    hu = np.interp(tu, t, h)
    dt = tu[1] - tu[0]
    n = len(tu)
    win = min(11, n if n % 2 == 1 else n - 1)
    if win <= POLY_DEG + 1:
        win = POLY_DEG + 3 if (POLY_DEG + 3) % 2 == 1 else POLY_DEG + 2
    hs = savgol_filter(hu, win, POLY_DEG)
    vs = savgol_filter(hu, win, POLY_DEG, deriv=1, delta=dt)
    ac = savgol_filter(hu, win, POLY_DEG, deriv=2, delta=dt)
    return {'h': CubicSpline(tu, hs), 'v': CubicSpline(tu, vs),
            'a': lambda tt, _c=CubicSpline(tu, ac): _c(tt) * 1000}


def hva(fit, t):
    """Height [km], speed [km/s] and acceleration [m/s^2] of a fitted track at t."""
    return fit['h'](t), fit['v'](t), fit['a'](t)


FIT_METHODS = {'Polynomial': fit_polynomial, 'Gallagher (2003)': fit_gallagher,
               'Byrne (2013)': fit_byrne}
FIT_COLOR = {'Polynomial': 'tab:blue', 'Gallagher (2003)': 'tab:green', 'Byrne (2013)': 'tab:red'}

_d = ALref[REF_TRACK]
_good = np.isfinite(_d['r_mean'])
t_fit = tg[_good].astype(float)
r_fit = _d['r_mean'][_good].astype(float)
se_r = _d['r_se'][_good].astype(float)
_pos = se_r[np.isfinite(se_r) & (se_r > 0)]
se_r = np.where(np.isfinite(se_r) & (se_r > 0), se_r, np.nanmedian(_pos) if _pos.size else 1e-3)
h_fit = r_fit * RS_KM
sig_h = se_r * RS_KM
t_dense = np.linspace(t_fit.min(), t_fit.max(), 400)

FIT_OUT = {}
rng = np.random.default_rng(1)
for name, fitter in FIT_METHODS.items():
    try:
        f0 = fitter(t_fit, h_fit, sig_h)
    except (RuntimeError, ValueError, TypeError) as ex:
        print(f'{name}: fit failed ({ex})')
        continue
    h0, v0, a0 = hva(f0, t_dense)
    boot = {'h': [], 'v': [], 'a': []}
    for _ in tqdm(range(N_BOOT), desc=name, leave=False):
        try:
            fb = fitter(t_fit, h_fit + rng.normal(0, sig_h), sig_h)
            hh, vv, aa = hva(fb, t_dense)
            boot['h'].append(hh)
            boot['v'].append(vv)
            boot['a'].append(aa)
        except (RuntimeError, ValueError):
            continue
    resid = f0['h'](t_fit) - h_fit
    dof = max(len(t_fit) - (POLY_DEG + 1), 1)
    chi2_red = float(np.nansum((resid / sig_h) ** 2) / dof)
    # chi2_red >> 1 means the fit misses the points by more than the quoted errors allow, so the
    # bootstrap band built from those errors is too narrow by roughly sqrt(chi2_red)
    scale = np.sqrt(max(chi2_red, 1.0)) if INFLATE_BY_CHI2 else 1.0
    FIT_OUT[name] = dict(
        h=h0 / RS_KM, v=v0, a=a0,
        h_sd=(np.nanstd(boot['h'], axis=0) / RS_KM if boot['h'] else np.zeros_like(t_dense)) * scale,
        v_sd=(np.nanstd(boot['v'], axis=0) if boot['v'] else np.zeros_like(t_dense)) * scale,
        a_sd=(np.nanstd(boot['a'], axis=0) if boot['a'] else np.zeros_like(t_dense)) * scale,
        chi2_red=chi2_red, scale=scale,
        rse_Rsun=np.sqrt(np.nansum(resid ** 2) / dof) / RS_KM)
    print(f'{name:18s} chi2_red = {chi2_red:8.1f}  ->  error bands scaled by {scale:.1f}')
print(f'{len(FIT_OUT)} of {len(FIT_METHODS)} methods converged')

# --- can each method actually describe this track? ------------------------------------------
# Converging is not the same as being applicable. A model whose functional form cannot represent
# the data will still return parameters; it just fits badly, and quoting its kinematics would be
# wrong. Both checks below are about the shape of the model, not the quality of the data.
APPLICABLE = {name: True for name in FIT_OUT}
if FIT_OUT:
    _best = min(d['chi2_red'] for d in FIT_OUT.values())
    for name, d in FIT_OUT.items():
        if d['chi2_red'] > max(10 * _best, 10.0):
            APPLICABLE[name] = False
            print(f'\n  *** WARNING: {name} fits this track far worse than the best method '
                  f'(chi2_red {d["chi2_red"]:.1f} vs {_best:.2f}, rms residual '
                  f'{d["rse_Rsun"]:.4f} Rsun). Do not quote its kinematics.')
    _g, _p = FIT_OUT.get('Gallagher (2003)'), FIT_OUT.get('Polynomial')
    if _g is not None and np.nanmin(_g['a']) >= 0 and _p is not None and np.nanmin(_p['a']) < 0:
        APPLICABLE['Gallagher (2003)'] = False
        print('\n  *** WARNING: the Gallagher et al. (2003) reciprocal-sum profile is '
              'POSITIVE-DEFINITE by construction -')
        print('      a(t) = [ (a_r e^(t/tau_r))^-1 + (a_d e^(-t/tau_d))^-1 ]^-1 with a_r, a_d > 0 '
              'is strictly > 0.')
        print('      It describes the impulsive ACCELERATION phase of an eruption and cannot '
              'represent a decelerating')
        print('      shock. The polynomial wants deceleration on this track, so the Gallagher '
              'curve is inapplicable here.')

In [ ]:
fig = plt.figure(figsize=[16, 4.8])
panels = [('h', r'$r\,/\,R_\odot$', '(a) Height'),
          ('v', r'$v_{\rm sh}$ [km s$^{-1}$]', '(b) Speed'),
          ('a', r'$a$ [m s$^{-2}$]', '(c) Acceleration')]
for i, (key, ylab, ttl) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 3, i)
    if key == 'h':
        # quote what the traced points actually are, not the word "SEM"
        ax.errorbar(t_fit, r_fit, yerr=se_r, fmt='o', ms=4, color='0.4', capsize=2, alpha=0.8,
                    zorder=1, label=(rf'traced: {len(t_fit)} pts, '
                                     rf'{r_fit.min():.3f}$-${r_fit.max():.3f} $R_\odot$, '
                                     rf'$\sigma$ = {np.mean(se_r):.4f} $R_\odot$'))
    for name, d in FIT_OUT.items():
        c = FIT_COLOR[name]
        ok = APPLICABLE.get(name, True)
        y, sd = d[key], d[key + '_sd']
        if key == 'h':
            lbl = (rf'{name}: rms {d["rse_Rsun"]:.4f} $R_\odot$, '
                   rf'$\chi^2_\nu$ = {d["chi2_red"]:.2f}')
        elif key == 'v':
            lbl = (f'{name}: {np.nanmin(y):.0f}$-${np.nanmax(y):.0f}, '
                   f'mean {np.nanmean(y):.0f} km s$^{{-1}}$')
        else:
            lbl = (f'{name}: {np.nanmin(y):+.0f} to {np.nanmax(y):+.0f}, '
                   f'mean {np.nanmean(y):+.0f} m s$^{{-2}}$')
        if not ok:
            lbl += '  [INAPPLICABLE]'
        ax.plot(t_dense, y, ls=('-' if ok else '--'), color=c, lw=1.8, label=lbl)
        ax.fill_between(t_dense, y - sd, y + sd, color=c, alpha=0.18)
    if key == 'a':
        ax.axhline(0, color='0.6', lw=0.8)
    ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
    ax.set_ylabel(ylab)
    ax.set_title(ttl)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7.5)
_scaled = 'bands widened by sqrt(chi2) ' if INFLATE_BY_CHI2 else 'bands are the raw bootstrap 1$\sigma$ '
fig.suptitle(f'Height-time fit comparison, {REF_TRACK} track ({REF_MODEL_NAME})\n'
             + _scaled + f'over {N_BOOT} refits; dashed = the model cannot describe this track',
             y=1.06, fontsize=12)
fig.tight_layout()
save_fig(fig, 'typeii_kinematics_fit_comparison')
plt.show()

In [ ]:
# goodness of fit and the grid-averaged kinematics each method implies. chi2_red is the primary
# metric (about 1 means the fit is consistent with the height errors); rse is the residual
# standard error in Rsun, i.e. the typical deviation of the fit from the traced heights
fit_compare = pd.DataFrame([{'method': name, 'applicable': APPLICABLE.get(name, True),
                             'chi2_red': d['chi2_red'],
                             'band_scale': d['scale'], 'rse_Rsun': d['rse_Rsun'],
                             'v_mean_kms': np.nanmean(d['v']), 'v_min_kms': np.nanmin(d['v']),
                             'v_max_kms': np.nanmax(d['v']), 'a_mean_ms2': np.nanmean(d['a'])}
                            for name, d in FIT_OUT.items()])
fit_compare.to_csv(os.path.join(OUTDIR, 'height_time_fit_comparison.csv'), index=False)
fit_compare.round(3)

> **Figure 7 &mdash; Height-time fit comparison.** `typeii_kinematics_fit_comparison.png`
>
> The same height-time track fitted three ways &mdash; a weighted polynomial, the Gallagher
> et&nbsp;al. (2003) two-phase acceleration profile, and Byrne et&nbsp;al. (2013) Savitzky-Golay
> smoothing &mdash; with *(a)* height, *(b)* speed and *(c)* acceleration. Every legend entry
> carries the numbers: the traced points give their count, height range and $\sigma$; each method
> gives its rms residual and $\chi^2_\nu$ in (a), its speed range and mean in (b), its acceleration
> range and mean in (c). Shaded bands are the raw parametric-bootstrap $1\sigma$.
>
> *A dashed line marked `[INAPPLICABLE]`* means the model's functional form cannot describe this
> track. The Gallagher profile in particular is positive-definite by construction and cannot
> represent a decelerating shock; since most type IIs decelerate past the impulsive phase, expect
> to see it flagged.
>
> **Table 6 &mdash; Fit comparison.** `height_time_fit_comparison.csv` &mdash; per method:
> applicability, $\chi^2_\nu$, the band scaling (1 unless `INFLATE_BY_CHI2` is on), the rms
> residual, and the speed and acceleration each method implies.

## A.8 &middot; Sensitivity to the density model &times; fold

**What "sweep" means here.** The entire chain from §A.4 &mdash; frequency to density to height to
$v_{\rm sh}$, $v_A$ and $B$ &mdash; is recomputed from the same traced lanes for every one of the
5&nbsp;models&nbsp;&times;&nbsp;4&nbsp;folds = 20 density profiles. `model_grid_sweep.csv` holds one
row per profile. Nothing about the tracing changes; only the assumed $n_e(r)$ does. The spread down
each column is therefore exactly the systematic error the density model contributes.

The band-split $X$ and $M_A$ are **model independent** and are quoted once per band. The height,
shock speed, Alfv&eacute;n speed and $B$ all depend on the assumed $n_e(r)$, so the spread across
the 5&nbsp;models&nbsp;&times;&nbsp;4&nbsp;folds grid *is* the systematic uncertainty and should be
quoted alongside the statistical error bars. Combinations that place the source below the solar
surface for this event return NaN and drop out of the grid rather than being extrapolated.

Note that the fundamental/harmonic pair gives no extra leverage here: since $f_H/2$ and $f_F$ map
to the same plasma density, both bands land on the same height in any given model. What the pair
does buy is **time coverage** &mdash; the joint track spans the whole burst, so the sweep is run on
it.

In [ ]:
sweep_rows = []
for name, model in tqdm(MODEL_GRID.items(), desc='model x fold'):
    agg = aggregate_lanes(passes, tg, model)
    if REF_TRACK not in agg:
        continue
    row = {'model': name}
    for key, lab in [('r', 'r_Rsun'), ('v', 'v_kms'), ('a', 'a_ms2'), ('vA', 'vA_kms'),
                     ('B', 'B_G'), ('ne', 'ne_cm3')]:
        row[lab], row[lab + '_se'] = grid_scalar(agg[REF_TRACK], key)
    sweep_rows.append(row)

sweep = pd.DataFrame(sweep_rows)
sweep['base'] = [m.rsplit(' x', 1)[0] for m in sweep['model']]
sweep['fold'] = [int(m.rsplit(' x', 1)[1]) for m in sweep['model']]
sweep.to_csv(os.path.join(OUTDIR, 'model_grid_sweep.csv'), index=False)
print(f'the full chain was recomputed on the {REF_TRACK} track for all {len(MODEL_GRID)} '
      'density profiles (5 models x 4 folds); only n_e(r) changed, the tracing did not')
print('the spread down each column IS the systematic error; compare it with the statistical '
      'errors in A.6 before quoting anything')
sweep[['model', 'r_Rsun', 'v_kms', 'a_ms2', 'vA_kms', 'B_G']].round(3)

> **Table 7 &mdash; Density-model sweep.** `model_grid_sweep.csv`
>
> The entire chain &mdash; frequency to density to height to $v_{\rm sh}$, $a$, $v_A$ and $B$
> &mdash; recomputed from the *same traced lanes* for each of the 5&nbsp;models&nbsp;$\times$&nbsp;4
> folds. Only the assumed $n_e(r)$ changes between rows; nothing about the tracing does.
>
> *The spread down each column is the systematic error the density model contributes*, and for this
> class of measurement it dominates the statistical error by one to two orders of magnitude. Quote
> the reference-model value with this range beside it.

In [ ]:
bases = list(BASE_MODELS.keys())
xpos = np.arange(len(bases))
fold_off = {f: (f - 2.5) * 0.16 for f in FOLDS}
fold_col = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:green', 4: 'tab:red'}

panels = [('r_Rsun', r'height $r$ [$R_\odot$]'),
          ('v_kms', r'shock speed $v_{\rm sh}$ [km s$^{-1}$]'),
          ('vA_kms', r'Alfv$\acute{\rm e}$n speed $v_A$ [km s$^{-1}$]'),
          ('B_G', r'$B$ [G]')]

fig = plt.figure(figsize=[14, 9])
for i, (col, ylab) in enumerate(panels, start=1):
    ax = fig.add_subplot(2, 2, i)
    for f in FOLDS:
        sub = sweep[sweep['fold'] == f].set_index('base').reindex(bases)
        ax.errorbar(xpos + fold_off[f], sub[col], yerr=sub[col + '_se'], fmt='o',
                    color=fold_col[f], ms=6, capsize=3, mec='k', mew=0.4,
                    label=(f'fold {f}' if i == 1 else None))
    ax.set_xticks(xpos)
    ax.set_xticklabels(bases, fontsize=8)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.3, axis='y')
    if i == 1:
        ax.legend(title='fold', fontsize=8, ncol=2)

fig.suptitle(f'Shock characteristics vs density model x fold ({REF_TRACK} track)',
             y=1.01, fontsize=13)
fig.tight_layout()
save_fig(fig, 'characteristics_vs_model_fold')
plt.show()

> **Figure 8 &mdash; Characteristics against density model and fold.**
> `characteristics_vs_model_fold.png`
>
> Height, shock speed, Alfv&eacute;n speed and magnetic field for every model&times;fold
> combination, colour-coded by fold. Error bars are the statistical (Monte-Carlo) errors of each
> combination.
>
> *Read the vertical scatter, not the individual points.* The points are not competing
> measurements; they are the same measurement under twenty different assumptions about the corona,
> and their spread is the honest uncertainty on any height-dependent quantity.

## A.9 &middot; Coronal magnetic field against empirical $B(r)$ laws

The band-split estimate $B=v_A\sqrt{\mu_0\rho}$ gives one point per model&times;fold, placed at the
height that model assigns. Comparing the cloud against published radial profiles shows whether the
type II is asking for a field strength that a normal corona can supply at that height. The
Gopalswamy & Yashiro (2011) law was calibrated between 6 and 23&nbsp;$R_\odot$ and is extrapolated
downwards here, so it should be read as an indication only.

In [ ]:
def B_dulk_mclean(r):
    """Dulk & McLean (1978); valid 1.02 <= r <= 10 Rsun."""
    r = np.asarray(r, float)
    return 0.5 * (r - 1.0) ** -1.5

def B_gopalswamy_yashiro(r):
    """Gopalswamy & Yashiro (2011); calibrated 6-23 Rsun, extrapolated below."""
    r = np.asarray(r, float)
    return 0.377 * r ** -1.25

def B_mann2023(r):
    """Mann et al. (2023), A&A 679, A64, Eq. 8, radial field [G]."""
    r = np.asarray(r, float)
    return 6.0 * r ** -3 + 1.18 * r ** -2


rr = np.linspace(1.05, 3.0, 300)
fig = plt.figure(figsize=[9, 6.5])
ax = fig.add_subplot(111)
ax.plot(rr, B_dulk_mclean(rr), 'k-', label='Dulk & McLean (1978)')
ax.plot(rr, B_gopalswamy_yashiro(rr), 'k--', label='Gopalswamy & Yashiro (2011), extrapolated')
ax.plot(rr, B_mann2023(rr), 'k:', label='Mann et al. (2023)')

mk = {'Newkirk': 'o', 'Saito': 's', 'Leblanc': '^', 'Baumbach-Allen': 'D', 'Mann 2023': 'v'}
for _, row in sweep.iterrows():
    if np.isfinite(row['r_Rsun']) and np.isfinite(row['B_G']):
        ax.errorbar(row['r_Rsun'], row['B_G'], yerr=row['B_G_se'], xerr=row['r_Rsun_se'],
                    fmt=mk[row['base']], color=fold_col[row['fold']], ms=8, capsize=2,
                    mec='k', mew=0.5, alpha=0.9)
if not np.isfinite(sweep['B_G']).any():
    ax.text(0.5, 0.5, 'no band-split estimate to plot: $B$ is NaN for every model.\n'
            '$B = v_A\\sqrt{\\mu_0\\rho}$ needs $M_A$, which is only defined for '
            '$1 \\leq X < 4$.\nCheck the upstream/downstream ordering printed in A.4.',
            transform=ax.transAxes, ha='center', va='center', fontsize=10, color='firebrick')
handles = [plt.Line2D([], [], marker=mk[b], color='0.4', ls='', mec='k', label=b) for b in bases]
handles += [plt.Line2D([], [], marker='o', color=fold_col[f], ls='', label=f'fold {f}') for f in FOLDS]
leg1 = ax.legend(loc='upper right', fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=handles, loc='lower left', fontsize=8, ncol=2, title='band-split estimate')
ax.set_yscale('log')
ax.set_xlabel(r'$r\,/\,R_\odot$')
ax.set_ylabel(r'$B$ [G]')
ax.set_title('Coronal magnetic field: band-split estimate vs empirical laws')
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
save_fig(fig, 'Bfield_comparison')
plt.show()

> **Figure 9 &mdash; Coronal magnetic field against empirical laws.** `Bfield_comparison.png`
>
> The band-split estimate $B=v_A\sqrt{\mu_0\rho}$, one point per model&times;fold at the height that
> model assigns (marker = model, colour = fold), against three published radial profiles.
> Gopalswamy & Yashiro (2011) was calibrated between 6 and 23&nbsp;$R_\odot$ and is extrapolated
> downwards here, so treat it as indicative only.
>
> *What it tests:* whether the type II is asking for a field strength a normal corona can supply at
> that height. Points lying far above every law would suggest the density jump, and hence $M_A$,
> has been overestimated.

## A.10 &middot; Characteristics table, LaTeX export and drafted results text

In [ ]:
rows = []
for b in BANDS_TRACED:
    s = scalars[b]
    tag = f'{BAND_NAME[b].lower()}, s={HARM[b]}'
    for key, lab, unit in [('drift_MHz_s', 'drift rate', 'MHz/s'),
                           ('rel_drift_s', 'relative drift (1/f)(df/dt)', '1/s'),
                           ('X', 'density jump X', '-'),
                           ('M_A', 'Alfven Mach number M_A', '-'),
                           ('rel_bandwidth', 'relative band split (f_U-f_L)/f_L', '-')]:
        rows.append({'track': b, 'quantity': f'{lab} ({tag})', 'value': s[key][0],
                     'error': s[key][1], 'unit': unit, 'note': 'model-independent'})

for key in TRACKS:
    d = ALref[key]
    if np.isfinite(d['r_mean']).sum() <= 2:
        continue
    tag = key
    fm = d['f_mean'][np.isfinite(d['f_mean'])]
    if fm.size:
        rows.append({'track': tag, 'quantity': f'frequency range ({tag})', 'value': np.nanmin(fm),
                     'error': np.nanmax(fm), 'unit': 'MHz (min, max)', 'note': 'observed'})
    for k, lab, unit in [('r', 'height r', 'Rsun'), ('v', 'shock speed', 'km/s'),
                         ('a', 'acceleration', 'm/s^2'), ('vA', 'Alfven speed', 'km/s'),
                         ('ne', 'upstream density n_e', 'cm^-3'),
                         ('B', 'magnetic field B', 'G')]:
        m, se = grid_scalar(d, k)
        rows.append({'track': tag, 'quantity': f'{lab} ({tag})', 'value': m, 'error': se,
                     'unit': unit, 'note': REF_MODEL_NAME})
    if REPORT_EXCITER_ENERGY:
        v_mean, _ = grid_scalar(d, 'v')
        rows.append({'track': tag, 'quantity': f'bulk exciter energy ({tag})',
                     'value': electron_energy_from_speed(v_mean) * 1e3, 'error': np.nan,
                     'unit': 'eV', 'note': f'{REF_MODEL_NAME}, E = (gamma-1) m_e c^2 at v_sh; '
                                           'NOT the energy of the emitting electrons'})

# the model x fold spread, i.e. the systematic error on every height-dependent quantity
for col, lab, unit in [('r_Rsun', 'height r', 'Rsun'), ('v_kms', 'shock speed', 'km/s'),
                       ('a_ms2', 'acceleration', 'm/s^2'),
                       ('vA_kms', 'Alfven speed', 'km/s'), ('B_G', 'magnetic field B', 'G')]:
    vals = sweep[col].to_numpy(float)
    vals = vals[np.isfinite(vals)]
    if vals.size:
        rows.append({'track': REF_TRACK,
                     'quantity': f'{lab} (model x fold range)', 'value': np.nanmin(vals),
                     'error': np.nanmax(vals), 'unit': f'{unit} (min, max)',
                     'note': 'systematic across the 20-model grid'})

char_table = pd.DataFrame(rows)
char_table.to_csv(os.path.join(OUTDIR, 'typeii_characteristics.csv'), index=False)


def df_to_latex(df, caption, label, float_fmt='{:.4g}'):
    """Minimal LaTeX table writer. Written by hand rather than via DataFrame.to_latex so it needs
    no jinja2, escapes the characters that appear in these tables, and emits a float-ready
    table environment."""
    def esc(x):
        if isinstance(x, float):
            return '' if not np.isfinite(x) else float_fmt.format(x)
        s = str(x)
        for a, b in [('\\', r'\textbackslash '), ('_', r'\_'), ('%', r'\%'), ('&', r'\&'),
                     ('#', r'\#'), ('^', r'\^{}'), ('~', r'\~{}')]:
            s = s.replace(a, b)
        return s
    cols = list(df.columns)
    out = ['\\begin{table}[htbp]', '\\centering',
           f'\\caption{{{caption}}}', f'\\label{{{label}}}',
           '\\begin{tabular}{' + 'l' * len(cols) + '}', '\\hline',
           ' & '.join(esc(c) for c in cols) + ' \\\\', '\\hline']
    for _, r in df.iterrows():
        out.append(' & '.join(esc(v) for v in r.tolist()) + ' \\\\')
    out += ['\\hline', '\\end{tabular}', '\\end{table}', '']
    return '\n'.join(out)


with open(os.path.join(OUTDIR, 'typeii_characteristics.tex'), 'w') as fh:
    fh.write(df_to_latex(char_table,
                         f'Characteristics of the type II radio burst observed by NenuFAR on '
                         f'{EVENT_DATE}, from the band-split fundamental and harmonic lanes. '
                         f'Model-independent quantities come from the band splits alone; the rest '
                         f'assume the {REF_MODEL_NAME} density model, with the range across the '
                         f'full model grid given separately.',
                         'tab:typeii_nenufar'))

picks = {'traces': tracer.traces, 'passes': passes, 'polarisation': pol_table,
         'fh_tests': fh_test, 'lane_order': LANE_ORDER, 'lane_role': LANE_ROLE,
         'config': {'N_REPS': N_REPS, 'TYPEII_WINDOW': TYPEII_WINDOW,
                    'TYPEII_FLIM': TYPEII_FLIM, 'HARM': HARM, 'POLY_DEG': POLY_DEG,
                    'FIT_IN_LOGF': FIT_IN_LOGF, 'REF_MODEL_NAME': REF_MODEL_NAME,
                    'REF_TRACK': REF_TRACK}}
with open(os.path.join(OUTDIR, 'typeii_picks.pkl'), 'wb') as fh:
    pickle.dump(picks, fh)

# anything in OUTDIR older than this run is left over from a previous version of the analysis.
# Mixing those into a paper is exactly the kind of mistake that is impossible to spot later.
stale = sorted(f for f in os.listdir(OUTDIR)
               if not f.startswith('.')
               and os.path.getmtime(os.path.join(OUTDIR, f)) < RUN_START)
print('written to', OUTDIR)
if stale:
    print(f'\n  *** WARNING: {len(stale)} file(s) in {OUTDIR} were NOT written by this run and '
          'are left over from an earlier one:')
    for f in stale:
        print(f'        {f}')
    print('      delete them, or move this run to a fresh OUTDIR, before using anything from '
          'that folder.')
char_table.round(4)

> **Table 8 &mdash; Burst characteristics.** `typeii_characteristics.csv`, `.tex`
>
> The paper-ready summary. Model-independent quantities first (drift rate, relative drift, density
> jump, Alfv&eacute;n Mach number, relative bandwidth, per band), then the model-dependent ones per
> lane on the reference model (frequency range, height, speed, acceleration, Alfv&eacute;n speed,
> upstream density, magnetic field), and finally the **model&times;fold range** of each
> height-dependent quantity as its systematic error. The `.tex` file is the same table as a
> `table` environment ready to `\input`.
>
> A warning is printed here listing any file in the output folder that this run did **not** write
> &mdash; leftovers from an earlier version of the analysis are otherwise impossible to spot later.

In [ ]:
# ---- drafted results text, filled with the numbers actually measured above ----
def _dp(e, min_dp=1, max_dp=5):
    """Decimal places that show the error to two significant figures."""
    if e is None or not np.isfinite(e) or e <= 0:
        return min_dp
    return int(np.clip(-np.floor(np.log10(abs(e))) + 1, min_dp, max_dp))


def _fmt(v, e, min_dp=1, sep=' +/- '):
    """Value +/- error with the precision set by the error, not by a fixed format."""
    if not np.isfinite(v):
        return 'n/a'
    nd = _dp(e, min_dp)
    if e is None or not np.isfinite(e) or e <= 0:
        return f'{v:.{nd}f}'
    return f'{v:.{nd}f}{sep}{e:.{nd}f}'


def _sci(v, nd=1):
    """Scientific notation as LaTeX inline maths."""
    if not np.isfinite(v) or v == 0:
        return 'n/a'
    ex = int(np.floor(np.log10(abs(v))))
    return f'${v / 10 ** ex:.{nd}f} \\times 10^{{{ex}}}$'


def _tex(v, e, min_dp=1):
    """The same as _fmt, wrapped as inline maths ready to paste into the manuscript."""
    if not np.isfinite(v):
        return 'n/a'
    nd = _dp(e, min_dp)
    if e is None or not np.isfinite(e) or e <= 0:
        return f'${v:.{nd}f}$'
    return f'${v:.{nd}f} \\pm {e:.{nd}f}$'


d = ALref[REF_TRACK]
r0, r0e = grid_scalar(d, 'r')
v0, v0e = grid_scalar(d, 'v')
vA0, vA0e = grid_scalar(d, 'vA')
B0, B0e = grid_scalar(d, 'B')
ne0, _ = grid_scalar(d, 'ne')
a0, a0e = grid_scalar(d, 'a')
X0, X0e = grid_scalar(d, 'X')
MA0, MA0e = grid_scalar(d, 'MA')
ta = (t0 + pd.Timedelta(seconds=float(np.nanmin(tg)))).strftime('%H:%M')
tb = (t0 + pd.Timedelta(seconds=float(np.nanmax(tg)))).strftime('%H:%M')

lines = [f'Type II radio burst, {EVENT_DATE}, NenuFAR', '=' * 78, '',
         f'observed {ta}-{tb} UT; one shock seen in both harmonics',
         f'every lane is analysed separately'
         + (f'; {JOINT} is the combined track' if MAKE_JOINT else ''), '']
for b in BANDS_TRACED:
    s_ = scalars[b]
    lines += [f'{BAND_NAME[b]} band (s = {HARM[b]})', '-' * 78,
              f'  drift rate                   {_fmt(*s_["drift_MHz_s"], min_dp=4)} MHz/s',
              f'  relative drift (1/f)(df/dt)  {_fmt(*s_["rel_drift_s"], min_dp=5)} 1/s',
              f'  relative band split          {_fmt(*s_["rel_bandwidth"], min_dp=3)}',
              f'  density jump X               {_fmt(*s_["X"], min_dp=2)}   (model-independent)',
              f'  Alfven Mach number M_A       {_fmt(*s_["M_A"], min_dp=2)}   (model-independent)']
    for lab in LANE_ORDER[b]:
        d = ALref.get(lab)
        if d is None:
            continue
        ff = _fits[lab]
        wa = (t0 + pd.Timedelta(seconds=float(ff['tmin']))).strftime('%H:%M:%S')
        wb_ = (t0 + pd.Timedelta(seconds=float(ff['tmax']))).strftime('%H:%M:%S')
        fm = d['f_mean'][np.isfinite(d['f_mean'])]
        pol = pol_table[pol_table['lane'] == lab]
        r_, r_e = grid_scalar(d, 'r')
        v_, v_e = grid_scalar(d, 'v')
        vA_, vA_e = grid_scalar(d, 'vA')
        B_, B_e = grid_scalar(d, 'B')
        a_, a_e = grid_scalar(d, 'a')
        ne_, _ = grid_scalar(d, 'ne')
        lines += ['', f'  {lab}  [{LANE_ROLE[lab]}]',
                  f'    traced                     {wa}-{wb_} UT over '
                  f'{np.nanmin(fm):.1f}-{np.nanmax(fm):.1f} MHz',
                  f'    height r                   {_fmt(r_, r_e, 2)} Rsun',
                  f'    upstream density n_e       {ne_:.3e} cm^-3',
                  f'    shock speed v_sh           {_fmt(v_, v_e, 0)} km/s',
                  f'    acceleration a             {_fmt(a_, a_e, 1)} m/s^2',
                  f'    Alfven speed v_A           {_fmt(vA_, vA_e, 0)} km/s',
                  f'    magnetic field B           {_fmt(B_, B_e, 2)} G',
                  f'    signed mean V/I            '
                  f'{pol["V_over_I_mean"].iloc[0]:+.4f} +/- {pol["V_over_I_sd"].iloc[0]:.4f} (sd)'
                  if len(pol) else '']
    lines += ['']

lines += [f'{REF_TRACK} on the model x fold grid ({REF_MODEL_NAME} is the reference)', '-' * 78,
          f'  height r      {np.nanmin(sweep["r_Rsun"]):.2f}-{np.nanmax(sweep["r_Rsun"]):.2f} Rsun',
          f'  shock speed   {np.nanmin(sweep["v_kms"]):.0f}-{np.nanmax(sweep["v_kms"]):.0f} km/s',
          f'  Alfven speed  {np.nanmin(sweep["vA_kms"]):.0f}-{np.nanmax(sweep["vA_kms"]):.0f} km/s',
          f'  magnetic field {np.nanmin(sweep["B_G"]):.3f}-{np.nanmax(sweep["B_G"]):.3f} G', '']

if len(fh_test):
    lines += ['Fundamental / harmonic consistency', '-' * 66]
    for _, row in fh_test.iterrows():
        nd = 5 if abs(row['value']) < 0.01 else 3        # the drifts are ~1e-3 per second
        exp = '' if not np.isfinite(row['expected']) else f'   (expected {row["expected"]:.{nd}f})'
        lines.append(f'  {row["test"]:34s} {_fmt(row["value"], row["error"], nd)}{exp}')

summary = '\n'.join(lines)
print(summary)

sF = scalars['F']
sH = scalars.get('H', sF)
_wF = _fits[SPLIT_PAIR['F'][0]]
_wH = _fits[SPLIT_PAIR['H'][0]] if 'H' in SPLIT_PAIR else _wF
para = (
    f"A type II radio burst was observed by NenuFAR on "
    f"{pd.Timestamp(EVENT_DATE).strftime('%d %B %Y')} between {ta} and {tb}~UT, in both the "
    f"fundamental and the harmonic of plasma emission. The fundamental drifted through the "
    f"observing band from "
    f"{(t0 + pd.Timedelta(seconds=float(_wF['tmin']))).strftime('%H:%M')}~UT and the harmonic, at "
    f"twice its frequency, entered the top of the band from "
    f"{(t0 + pd.Timedelta(seconds=float(_wH['tmin']))).strftime('%H:%M')}~UT and followed the same "
    f"shock to the end of the event; over the interval in which both were visible their frequency "
    f"ratio is "
    f"{_tex(float(fh_test.iloc[0]['value']), float(fh_test.iloc[0]['error']), 2) if len(fh_test) else 'n/a'}, "
    f"confirming the identification. The relative drift rate is "
    f"{_tex(*sF['rel_drift_s'], min_dp=4)}~s$^{{-1}}$. Both bands are separately band-split, "
    f"giving relative bandwidths of {_tex(*sF['rel_bandwidth'], min_dp=2)} (F) and "
    f"{_tex(*sH['rel_bandwidth'], min_dp=2)} (H). Interpreting the split as emission from the "
    f"upstream and downstream sides of the shock front gives a density jump, averaged over the "
    f"two bands, of "
    f"$X = {_tex(X0, X0e, 2)[1:-1]}$ and, through the Rankine-Hugoniot relation for a perpendicular "
    f"shock, an Alfv\\'en Mach number of $M_A = {_tex(MA0, MA0e, 2)[1:-1]}$; neither depends on the "
    f"assumed coronal density model. Because $f_H/2$ and $f_F$ map to the same local density, the "
    f"two bands were combined into a single height-time track spanning the whole burst. Adopting a "
    f"{REF_MODEL_NAME.replace(' x', ', fold-')} electron-density profile places the source at "
    f"$r = {_tex(r0, r0e, 2)[1:-1]}\\,R_\\odot$, where the upstream density is "
    f"{_sci(ne0)}~cm$^{{-3}}$, and yields a shock speed of "
    f"$v_{{\\rm sh}} = {_tex(v0, v0e, 0)[1:-1]}$~km~s$^{{-1}}$. Repeating the analysis over a grid "
    f"of five density models at folds 1--4 shifts the height to "
    f"{np.nanmin(sweep['r_Rsun']):.2f}--{np.nanmax(sweep['r_Rsun']):.2f}~$R_\\odot$ and the speed "
    f"to {np.nanmin(sweep['v_kms']):.0f}--{np.nanmax(sweep['v_kms']):.0f}~km~s$^{{-1}}$, which we "
    f"adopt as the systematic uncertainty. The implied upstream Alfv\\'en speed is "
    f"$v_A = v_{{\\rm sh}}/M_A = {_tex(vA0, vA0e, 0)[1:-1]}$~km~s$^{{-1}}$, and combining it with "
    f"the upstream density gives a coronal magnetic field of $B = {_tex(B0, B0e, 2)[1:-1]}$~G "
    f"({np.nanmin(sweep['B_G']):.2f}--{np.nanmax(sweep['B_G']):.2f}~G across the model grid) at "
    f"that height, in line with the empirical radial profiles of \\citet{{dulk1978}} and "
    f"\\citet{{mann2023}}."
)

with open(os.path.join(OUTDIR, 'typeii_results_text.md'), 'w') as fh:
    fh.write(f'# Type II results, NenuFAR {EVENT_DATE}\n\n'
             '## Measured values\n\n```\n' + summary + '\n```\n\n'
             '## Draft paragraph (LaTeX-ready)\n\n' + para + '\n')

print('\n' + '=' * 78 + '\nDRAFT PARAGRAPH (LaTeX-ready)\n' + '=' * 78)
print(textwrap.fill(para, width=95))
print('\nsaved', os.path.join(OUTDIR, 'typeii_results_text.md'))

> **Output 2 &mdash; Drafted results text.** `typeii_results_text.md`
>
> The measured values laid out per band and per lane, followed by a LaTeX-ready paragraph with the
> numbers already substituted, errors quoted to two significant figures with the value matched, and
> `\citet{}` keys in place. Also written: `typeii_picks.pkl`, holding the raw traces, the passes,
> the lane roles and the configuration, so any figure can be regenerated without re-tracing.

## A.11 &middot; Conventions and caveats

- **One burst, one shock, two harmonics.** The bands occupy different intervals of the observing
  window only because $f_H=2f_F$ carries the harmonic across the top of the band while the
  fundamental is on its way out of the bottom; they are simultaneous emission from the same shock.
- **Lanes are named, not declared.** Each trace is recorded as `F lane 1`, `H lane 1` and so on in
  the order you make it. &sect;A.4 then sorts the lanes of each band in frequency and takes the
  lowest as the **upstream (unshocked)** branch and the next as the **downstream (compressed)**
  branch, so $X=(f_U/f_L)^2\ge1$. Heights, $n_e$ and $B$ are quoted from the upstream branch. Extra
  lanes of a band are kept and reported but do not enter the split.
- **$X$ and $M_A$ carry no density model** and are measured twice, once from each band's split;
  they should agree, and &sect;A.5 checks that they do. Height, $v_{\rm sh}$, $v_A$ and $B$ all do
  depend on the model, and the 5&times;4 grid in &sect;A.8 is the systematic error on them. Quote
  the reference-model value with the grid range beside it.
- **The `F+H joint` track is the one to quote.** It averages the two bands' heights where they
  overlap and takes whichever is available elsewhere, so it covers the whole burst. The bands add
  no independent constraint on the density model &mdash; $f_H/2$ and $f_F$ give the same
  density &mdash; but they do extend the time baseline, which is what a second derivative needs.
- **$M_A$ assumes a perpendicular shock** with $\gamma=5/3$. That is the standard band-split
  assumption but it is an assumption; an oblique geometry lowers the inferred $M_A$ at fixed $X$.
- **Lane fits are in $\log_{10}f$** (`FIT_IN_LOGF`). Over an octave of frequency a quadratic in $f$
  is much too stiff and biases the drift rate and the split; the relative drift then follows as
  $\ln 10\,\mathrm{d}\log_{10}f/\mathrm{d}t$.
- **Error bars combine two sources**: the spread between your repeats of a lane (reproducibility)
  and the polynomial coefficient covariance (fitting). The Monte-Carlo pools both; `_sd` is the
  total spread and `_se` divides it by $\sqrt{N_{\rm pass}}$. With `N_REPS = 1` only the fitting
  error is present, so trace each lane at least twice if you want an honest error bar.
- **Acceleration is in m&nbsp;s$^{-2}$** everywhere (speed in km&nbsp;s$^{-1}$, height in
  $R_\odot$). It is a second derivative of a fitted curve, so &sect;A.7 exists precisely to show how
  much of it depends on the fitting choice.
- **Kinematics are gated**: a model must constrain more than a quarter of the grid with a resolved
  height change, and only outward speeds below 3000&nbsp;km&nbsp;s$^{-1}$ are kept; the derivatives
  are blanked outside the span each track was actually traced over. Model&times;fold combinations
  that put the source at or below the surface therefore drop out rather than producing a spurious
  number.
- **The "bulk exciter energy"** is $(\gamma-1)m_ec^2$ evaluated at the shock speed. For a type II
  this is the kinetic energy of an electron moving with the shock, not the energy of the
  beam-accelerated electrons that produce the emission, and it is firmly non-relativistic.
- **Polarisation is supporting evidence.** The expectation that fundamental emission is more
  strongly circularly polarised than harmonic emission holds statistically, but type II
  polarisation is weak and its sense depends on the line-of-sight field, so &sect;A.5 weighs it
  alongside the frequency-ratio, relative-drift and height-agreement tests rather than on its own.
- **Display stretch.** `LAYER_PLO` clips the bottom of the colour scale and `LAYER_GAMMA` stretches
  the faint end; the comparison figure in &sect;A.1 exists because how much fine structure you can
  see, and therefore trace, depends on both. Nothing is averaged in frequency.
- **Fine structure** (herringbones, striae) can be traced as extra lanes of a band, but the fits
  are low-order curves through whatever you click, so they describe the envelope rather than the
  individual striae. Zoom in with the toolbar before tracing them, and raise `LAYER_MAX_T` if the
  time decimation is smoothing them away.
- **All plots use `pcolormesh`**, never `imshow`, and no tick labels are rotated.

**Outputs** (in `OUTDIR`): `typeii_characteristics.csv` / `.tex`, `typeii_polarisation.csv`,
`fundamental_harmonic_tests.csv`, `height_time_fit_comparison.csv`, `model_grid_sweep.csv`,
`typeii_results_text.md`, `typeii_picks.pkl`, and every figure as a 300&nbsp;dpi PNG.

## A.12 &middot; Methods, in the order they are applied

A compact description of the whole chain, suitable for adapting into a paper's methods section.

**Observations and pre-processing.** NenuFAR high-resolution Stokes&nbsp;I and V/I dynamic spectra
are used. Stokes&nbsp;I is converted to decibels on load; the tracing layer is that frame minus the
time median of each frequency channel, which removes the instrumental bandpass and the quiet-Sun
background and leaves the burst as an excess above zero. The layer is restricted to the type II
window and block-averaged in time only, never in frequency, since the band splitting and fine
structure are frequency-domain features.

**Lane tracing.** The emission lanes are traced by hand on a single rendering of the dynamic
spectrum, either by clicking points along a lane or by placing a B&eacute;zier curve on it with
sliders. Each lane is traced several times so the spread between repeats supplies a reproducibility
error; repeats produced by jittering one B&eacute;zier's control points are recorded as such,
because their spread measures that jitter rather than the reproducibility of the tracing. Each
traced point is assigned a frequency uncertainty equal to the rms offset between the trace and the
local intensity maximum, and densely sampled curves are thinned to one point per `FIT_MIN_DT_S`
seconds so that correlated samples do not inflate the apparent information content.

**Lane fits.** Each lane is fitted with a low-order polynomial in $\log_{10}f$ against time, the
coefficient covariance being computed from the assigned frequency uncertainties rather than from
the residual scatter. Log space is used because a type II lane spans close to an octave, over which
a polynomial in $f$ is too stiff. The drift rate follows analytically as
$\dot f=f\ln10\;\mathrm{d}\log_{10}f/\mathrm{d}t$, and the relative drift $\dot f/f$ is independent
of the harmonic number and therefore directly comparable between bands.

**Band identification.** The lanes of each band are ordered by frequency over the interval in which
they overlap in time, the lower being taken as the upstream (unshocked) branch of the split. The
fundamental/harmonic assignment is then tested against the data: the frequency ratio over the
overlap must be 2, the relative drifts must agree, the two bands must map to the same height, and
the degree of circular polarisation is compared between them.

**Shock diagnostics.** The band split gives the density jump $X=(f_U/f_L)^2$ and, through the
Rankine-Hugoniot relation for a perpendicular shock with $\gamma=5/3$,
$M_A=\sqrt{X(X+5)/[2(4-X)]}$. Neither involves a density model, and each band provides an
independent determination. Frequencies are converted to local electron density through
$n_e=(f/s)^2/\mathrm{const}$ with $s$ the harmonic number, and to height by numerically inverting
an assumed $n_e(r)$; out-of-range densities return no height rather than an extrapolation.
Differentiating $r(t)$ gives the shock speed and acceleration, and
$v_A=v_{\rm sh}/M_A$, $B=v_A\sqrt{\mu_0\rho}$ with $\rho=\mu m_p n_e$ complete the chain.

**Uncertainties.** Statistical errors are propagated by Monte Carlo: for each traced repeat, the
lane's polynomial coefficients are drawn many times from their covariance and each draw is pushed
through the entire chain, so the fitting and reproducibility errors combine. The height-time fits
carry a separate parametric bootstrap in which every height is displaced by a draw from its own
$1\sigma$ before refitting. Systematic error is assessed by recomputing everything for five
electron-density models at folds 1&ndash;4; for height-dependent quantities that systematic exceeds
the statistical error by one to two orders of magnitude and is the one that should be quoted.

**Checks applied before any number is used.** Traces are verified to sit on emission rather than on
blank spectrum; quantities that must agree are compared with their combined errors and flagged
beyond a set significance; accelerations are reported only where the traced baseline can constrain
a curvature, and are labelled as model-dominated; and each height-time model is tested for whether
its functional form can represent the track at all before its kinematics are quoted.